# Шаг 1. Импорты и загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML библиотеки
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
from scipy.stats import pearsonr, spearmanr, mannwhitneyu, kruskal, shapiro, ttest_rel, f_oneway

import xgboost as xgb
import lightgbm as lgb
import optuna
import joblib
import json

# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Конфигурация
DATA_PATH = Path("D:/denis/APP/app_new/data")
DATA_PATH.mkdir(exist_ok=True)
plots_path = DATA_PATH / "plots"
plots_path.mkdir(exist_ok=True)

print("="*80)
print("📊 АНАЛИЗ ASO ДЛЯ ПРИЛОЖЕНИЯ SOUND AMPLIFIER")
print("="*80)
print(f"Путь к данным: {DATA_PATH}")
print(f"Путь к графикам: {plots_path}")

📊 АНАЛИЗ ASO ДЛЯ ПРИЛОЖЕНИЯ SOUND AMPLIFIER
Путь к данным: D:\denis\APP\app_new\data
Путь к графикам: D:\denis\APP\app_new\data\plots


In [2]:
# =====================================================
# ШАГ 1: ЗАГРУЗКА, ПРЕДОБРАБОТКА И ДЕДУПЛИКАЦИЯ (ОПТИМИЗИРОВАНО)
# =====================================================
print("\n" + "="*80)
print("ШАГ 1: ЗАГРУЗКА, ПРЕДОБРАБОТКА И ДЕДУПЛИКАЦИЯ ДАННЫХ")
print("="*80)

import time
start_time = time.time()

# =====================================================
# ФУНКЦИЯ ПАРСИНГА EXCEL
# =====================================================
def parse_excel_file(file_path, year):
    print(f"\nПарсинг файла: {file_path.name}")
    df_raw = pd.read_excel(file_path, header=None)
    
    # Извлечение дат из строки 0
    date_cols = {}
    for col_idx, val in df_raw.iloc[0, :].items():
        if isinstance(val, (pd.Timestamp, datetime)):
            date_cols[col_idx] = val
        elif isinstance(val, str) and val.startswith('202'):
            try:
                date_cols[col_idx] = pd.to_datetime(val)
            except:
                pass
    
    print(f"  - Найдено дат: {len(date_cols)}")
    
    # Данные US из строки 2 (органика и активации - общие для всех ключей)
    us_organic_cols = {}
    us_activation_cols = {}
    us_row = 2
    
    if us_row < len(df_raw):
        for col_idx, date in date_cols.items():
            if col_idx < len(df_raw.columns):
                val = df_raw.iloc[us_row, col_idx]
                try:
                    if not pd.isna(val) and str(val).strip() not in ['', '–', '-']:
                        num_val = float(val)
                        
                        next_col = col_idx + 1
                        if next_col < len(df_raw.columns):
                            next_val = df_raw.iloc[us_row, next_col]
                            next_has_data = not pd.isna(next_val) and str(next_val).strip() not in ['', '–', '-']
                            
                            if next_has_data:
                                try:
                                    next_num_val = float(next_val)
                                    us_activation_cols[date] = num_val
                                    us_organic_cols[date_cols[next_col]] = next_num_val
                                except:
                                    us_organic_cols[date] = num_val
                            else:
                                us_organic_cols[date] = num_val
                        else:
                            us_organic_cols[date] = num_val
                except:
                    pass
    
    print(f"  - Найдено органики US: {len(us_organic_cols)}")
    print(f"  - Найдено активаций US: {len(us_activation_cols)}")
    
    # Парсинг ключевых слов (начинаются со строки 4)
    keyword_start_row = 4
    keyword_end_row = len(df_raw)
    
    for idx in range(keyword_start_row, min(keyword_start_row + 500, len(df_raw))):
        val = df_raw.iloc[idx, 2]
        if pd.isna(val) or str(val).strip() == '':
            keyword_end_row = idx
            break
        if isinstance(val, str):
            val_lower = val.lower()
            if 'лимит' in val_lower or 'итого' in val_lower or 'total' in val_lower:
                keyword_end_row = idx
                break
    
    print(f"  - Диапазон ключевых слов: {keyword_start_row} - {keyword_end_row}")
    
    # Сбор данных
    keywords_data = []
    for row_idx in range(keyword_start_row, keyword_end_row):
        keyword = df_raw.iloc[row_idx, 2]
        if pd.isna(keyword) or str(keyword).strip() == '':
            continue
        keyword_str = str(keyword).strip()
        if len(keyword_str) < 2:
            continue
        
        for col_idx, date in date_cols.items():
            if col_idx + 1 >= len(df_raw.columns):
                continue
            motiv_val = df_raw.iloc[row_idx, col_idx]
            position_val = df_raw.iloc[row_idx, col_idx + 1]
            
            has_motiv = not pd.isna(motiv_val) and str(motiv_val).strip() not in ['', '–', '-']
            has_position = not pd.isna(position_val) and str(position_val).strip() not in ['', '–', '-']
            
            if not has_motiv and not has_position:
                continue
            
            try:
                motiv = float(motiv_val) if has_motiv else 0
            except:
                motiv = 0
            try:
                position = float(position_val) if has_position else -1
            except:
                position = -1
            
            keywords_data.append({
                'date': date,
                'keyword': keyword_str,
                'motiv': motiv,
                'position': position,
                'year': year,
                'organic_us': us_organic_cols.get(date, 0),
                'activation_us': us_activation_cols.get(date, 0)
            })
    
    df_result = pd.DataFrame(keywords_data)
    print(f"  - Загружено {len(df_result):,} записей")
    print(f"  - Уникальных ключей: {df_result['keyword'].nunique()}")
    
    if len(df_result) > 0:
        print(f"  - Период: {df_result['date'].min()} - {df_result['date'].max()}")
    
    return df_result


# =====================================================
# ЗАГРУЗКА ФАЙЛОВ
# =====================================================
FILES = {
    2024: DATA_PATH / "Sound Amplifier_2024.xlsx",
    2025: DATA_PATH / "Sound Amplifier_2025.xlsx",
    2026: DATA_PATH / "Sound Amplifier_2026.xlsx"
}

all_dfs = []
for year, file_path in FILES.items():
    if file_path.exists():
        df_year = parse_excel_file(file_path, year)
        if len(df_year) > 0:
            all_dfs.append(df_year)
    else:
        print(f"\n⚠️ Файл {file_path} не найден")

# Объединение
df = pd.concat(all_dfs, ignore_index=True)

print("\n" + "="*60)
print("ДО ДЕДУПЛИКАЦИИ:")
print(f"  - Всего записей: {len(df):,}")
print(f"  - Уникальных ключей: {df['keyword'].nunique()}")
print(f"  - Период: {df['date'].min()} - {df['date'].max()}")
print("="*60)

# =====================================================
# ДЕДУПЛИКАЦИЯ
# =====================================================
print("\n" + "="*60)
print("ДЕДУПЛИКАЦИЯ ДАННЫХ")
print("="*60)

print("\nПроверка пересечений по периодам:")
for year1, file1 in FILES.items():
    for year2, file2 in FILES.items():
        if year1 < year2:
            df1 = all_dfs[list(FILES.keys()).index(year1)]
            df2 = all_dfs[list(FILES.keys()).index(year2)]
            if len(df1) > 0 and len(df2) > 0:
                overlap = set(df1['date'].dt.date).intersection(set(df2['date'].dt.date))
                if overlap:
                    print(f"  {year1} и {year2}: пересечение {len(overlap)} дней")

print(f"\nДо дедупликации: {len(df):,} записей")
duplicates = df.duplicated(subset=['date', 'keyword'], keep=False).sum()
print(f"Найдено дублирующихся записей: {duplicates:,}")

df = df.drop_duplicates(subset=['date', 'keyword'], keep='first')
print(f"✅ Удалены дубликаты (оставлена первая запись)")

print(f"После дедупликации: {len(df):,} записей")
unique_check = df.duplicated(subset=['date', 'keyword']).sum()
print(f"Осталось дубликатов: {unique_check}")

df = df.sort_values(['keyword', 'date']).reset_index(drop=True)

print("\n" + "="*60)
print("ИТОГО ПОСЛЕ ДЕДУПЛИКАЦИИ:")
print(f"  - Всего записей: {len(df):,}")
print(f"  - Уникальных ключей: {df['keyword'].nunique()}")
print(f"  - Период: {df['date'].min()} - {df['date'].max()}")
print("="*60)

# =====================================================
# ПРЕДОБРАБОТКА
# =====================================================
print("\nОБРАБОТКА ПРОПУСКОВ:")
df['motiv'] = df['motiv'].fillna(0)
df['organic_us'] = df['organic_us'].fillna(0)
df['activation_us'] = df['activation_us'].fillna(0)
df['position'] = df['position'].fillna(-1)
print("  + Все пропуски обработаны")

# =====================================================
# БАЗОВЫЕ ПРИЗНАКИ
# =====================================================
print("\nСОЗДАНИЕ БАЗОВЫХ ПРИЗНАКОВ:")
t0 = time.time()

# Временные признаки
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
print("  + Временные признаки добавлены")

# Группировка (кэшируем)
grouped = df.groupby('keyword', sort=False)

# Лаги
for lag in [1, 2, 3, 7, 14, 30]:
    df[f'motiv_lag_{lag}'] = grouped['motiv'].shift(lag).fillna(0)
    df[f'position_lag_{lag}'] = grouped['position'].shift(lag).fillna(-1)
print("  + Лаговые признаки добавлены (1, 2, 3, 7, 14, 30)")

# Скользящие средние
for window in [3, 7, 14, 30]:
    df[f'motiv_ma_{window}'] = grouped['motiv'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()).fillna(0)
    df[f'position_ma_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()).fillna(-1)
    df[f'motiv_std_{window}'] = grouped['motiv'].transform(
        lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
    df[f'position_std_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
print("  + Скользящие средние добавлены (3, 7, 14, 30)")

# =====================================================
# РАСШИРЕННЫЕ СТАТИСТИЧЕСКИЕ ПРИЗНАКИ
# =====================================================
print("\nСОЗДАНИЕ РАСШИРЕННЫХ СТАТИСТИЧЕСКИХ ПРИЗНАКОВ:")

# EWM
for span in [3, 7, 14, 30]:
    df[f'motiv_ewm_{span}'] = grouped['motiv'].transform(
        lambda x: x.ewm(span=span, adjust=False).mean()).fillna(0)
    df[f'position_ewm_{span}'] = grouped['position'].transform(
        lambda x: x.ewm(span=span, adjust=False).mean()).fillna(-1)
print("  + Экспоненциальные скользящие средние (EWM)")

# Медианы
for window in [3, 7, 14]:
    df[f'motiv_median_{window}'] = grouped['motiv'].transform(
        lambda x: x.rolling(window, min_periods=1).median()).fillna(0)
    df[f'position_median_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).median()).fillna(-1)
print("  + Медианные скользящие")

# Min/Max
for window in [7, 14, 30]:
    df[f'motiv_min_{window}'] = grouped['motiv'].transform(
        lambda x: x.rolling(window, min_periods=1).min()).fillna(0)
    df[f'motiv_max_{window}'] = grouped['motiv'].transform(
        lambda x: x.rolling(window, min_periods=1).max()).fillna(0)
    df[f'position_min_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).min()).fillna(-1)
    df[f'position_max_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).max()).fillna(-1)
print("  + Min/Max за период (7, 14, 30)")

# Ранги
df['position_rank'] = df.groupby('date')['position'].rank(pct=True).fillna(0)
df['motiv_rank'] = df.groupby('date')['motiv'].rank(pct=True).fillna(0)
print("  + Ранги позиции и мотива")

# Процентные изменения
for period in [1, 3, 7, 14]:
    df[f'position_change_{period}d'] = grouped['position'].pct_change(period).replace([np.inf, -np.inf], 0).fillna(0)
    df[f'motiv_change_{period}d'] = grouped['motiv'].pct_change(period).replace([np.inf, -np.inf], 0).fillna(0)
print("  + Процентные изменения (1, 3, 7, 14 дней)")

# Волатильность
for window in [7, 14, 30]:
    df[f'position_volatility_{window}'] = grouped['position'].transform(
        lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
print("  + Волатильность позиции")

# Отношения
df['motiv_position_ratio'] = (df['motiv'] / (df['position'] + 1)).replace([np.inf, -np.inf], 0).fillna(0)
print("  + Отношение мотива к позиции")

# Тренды (оптимизированно через raw=True)
print("  + Расчёт трендов (оптимизировано)...")
df['position_trend_7d'] = grouped['position'].transform(
    lambda x: x.rolling(7, min_periods=2).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if len(y) >= 2 else 0,
        raw=True
    )
).fillna(0)

df['motiv_trend_7d'] = grouped['motiv'].transform(
    lambda x: x.rolling(7, min_periods=2).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if len(y) >= 2 else 0,
        raw=True
    )
).fillna(0)
print("  + Тренды позиции и мотива")

# Статистика органики/активаций
for window in [7, 14, 30]:
    df[f'organic_ma_{window}'] = grouped['organic_us'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()).fillna(0)
    df[f'activation_ma_{window}'] = grouped['activation_us'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()).fillna(0)
    df[f'organic_std_{window}'] = grouped['organic_us'].transform(
        lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
    df[f'activation_std_{window}'] = grouped['activation_us'].transform(
        lambda x: x.rolling(window, min_periods=1).std()).fillna(0)
print("  + Статистика органики и активаций")

# Разницы со средними
df['motiv_ma_diff'] = df['motiv'] - df['motiv_ma_7']
df['position_ma_diff'] = df['position'] - df['position_ma_7']
print("  + Разницы со скользящими средними")

# Сценарии (ВЕКТОРИЗОВАНО)
df['motiv_change'] = grouped['motiv'].diff().fillna(0)
df['position_change'] = grouped['position'].diff().fillna(0)
df['motiv_pct_change'] = grouped['motiv'].pct_change().replace([np.inf, -np.inf], 0).fillna(0)
df['position_pct_change'] = grouped['position'].pct_change().replace([np.inf, -np.inf], 0).fillna(0)

motiv_pct = df['motiv_pct_change'].values
scenarios = np.full(len(df), 'stable', dtype=object)
scenarios[motiv_pct > 0] = 'gentle_growth'
scenarios[motiv_pct >= 0.3] = 'sharp_growth'
scenarios[motiv_pct < 0] = 'gentle_decline'
scenarios[motiv_pct <= -0.3] = 'sharp_decline'
df['motiv_scenario'] = scenarios
print("  + Сценарии изменения мотива (векторизовано)")

# Целевые позиции
df['target_40'] = (df['position'] <= 40).astype(int)
df['target_20'] = (df['position'] <= 20).astype(int)
df['target_10'] = (df['position'] <= 10).astype(int)
print("  + Целевые позиции (40, 20, 10)")

# Метрики роста
for h in [1, 2, 3, 7, 14]:
    df[f'position_delta_{h}d'] = grouped['position'].shift(-h) - df['position']
    df[f'position_delta_{h}d'] = df[f'position_delta_{h}d'].fillna(0)
print("  + Метрики роста (1, 2, 3, 7, 14 дней)")

print(f"  ⏱️  Время создания признаков: {time.time() - t0:.1f} сек")

# =====================================================
# 🆕 ПРИЗНАКИ ФРИЗОВ И ОБНОВЛЕНИЙ iOS (ИСПРАВЛЕНО)
# =====================================================
print("\nСОЗДАНИЕ ПРИЗНАКОВ ФРИЗОВ И ОБНОВЛЕНИЙ iOS:")
t1 = time.time()

# =====================================================
# СПРАВОЧНИКИ
# =====================================================
FREEZE_PERIODS = [
    ('2025-01-04', '2025-01-08', 'Новогодний фриз 2025'),
    ('2025-02-25', '2025-03-10', 'Февральский фриз'),
    ('2025-03-15', '2025-03-15', 'Короткий фриз'),
    ('2025-05-05', '2025-05-12', 'Майский фриз'),
    ('2025-08-15', '2025-08-30', 'Августовский фриз'),
    ('2025-09-10', '2025-09-16', 'Сентябрьский фриз'),
    ('2025-09-28', '2025-10-31', 'Осенний фриз'),
    ('2026-01-06', '2026-01-19', 'Новогодний фриз 2026'),
    ('2026-04-07', '2026-04-21', 'Апрельский фриз'),
    ('2026-06-29', '2026-07-18', 'Летний фриз'),
]

IOS_UPDATES = [
    ('2025-03-31', '18.4'),
    ('2025-05-12', '18.5'),
    ('2025-07-29', '18.6'),
    ('2025-09-15', '18.7/26.0'),
    ('2025-11-03', '26.1'),
    ('2025-12-12', '26.2'),
    ('2026-02-11', '26.3'),
    ('2026-03-24', '26.4'),
    ('2026-03-31', '26.4.1'),
    ('2026-05-11', '26.5'),
    ('2026-06-01', '26.5.1'),
    ('2026-06-29', '26.5.2'),
    ('2026-07-27', '26.6'),
    ('2026-08-17', '26.6.1'),
]

# =====================================================
# 1. ПРИЗНАК is_freeze
# =====================================================
df['is_freeze'] = 0
df['freeze_name'] = ''

for start_str, end_str, name in FREEZE_PERIODS:
    start = pd.to_datetime(start_str)
    end = pd.to_datetime(end_str)
    mask = (df['date'] >= start) & (df['date'] <= end)
    df.loc[mask, 'is_freeze'] = 1
    df.loc[mask, 'freeze_name'] = name

print(f"  + Признак is_freeze: {df['is_freeze'].sum():,} записей в фризах")

# =====================================================
# 2. ВЕКТОРИЗОВАННЫЙ РАСЧЁТ days_to_freeze и days_after_freeze
# =====================================================
# Собираем все даты фризов
freeze_dates_list = []
for start_str, end_str, _ in FREEZE_PERIODS:
    start = pd.to_datetime(start_str)
    end = pd.to_datetime(end_str)
    freeze_dates_list.extend(pd.date_range(start, end).tolist())

freeze_dates_list = sorted(set(freeze_dates_list))

# Конвертируем в numpy datetime64[D]
freeze_dates_np = np.array([
    pd.Timestamp(d).to_datetime64().astype('datetime64[D]') 
    for d in freeze_dates_list
], dtype='datetime64[D]')

df_dates_np = df['date'].values.astype('datetime64[D]')

# ---------- days_to_freeze (дни ДО следующего фриза) ----------
diff_future = freeze_dates_np[None, :] - df_dates_np[:, None]
diff_future_f = diff_future.astype(np.float64)
mask_future = diff_future_f > 0
diff_future_masked = np.where(mask_future, diff_future_f, np.inf)

df['days_to_freeze'] = np.min(diff_future_masked, axis=1)
df['days_to_freeze'] = np.where(np.isinf(df['days_to_freeze']), 999, df['days_to_freeze'])
df['days_to_freeze'] = df['days_to_freeze'].astype(int)

# ---------- days_after_freeze (дни ПОСЛЕ последнего фриза) ----------
diff_past = df_dates_np[:, None] - freeze_dates_np[None, :]
diff_past_f = diff_past.astype(np.float64)
mask_past = diff_past_f > 0
diff_past_masked = np.where(mask_past, diff_past_f, np.inf)

df['days_after_freeze'] = np.min(diff_past_masked, axis=1)
df['days_after_freeze'] = np.where(np.isinf(df['days_after_freeze']), 999, df['days_after_freeze'])
df['days_after_freeze'] = df['days_after_freeze'].astype(int)

print(f"  + Признаки days_to_freeze, days_after_freeze добавлены (векторизовано)")

# =====================================================
# 3. iOS ОБНОВЛЕНИЯ
# =====================================================
df['is_ios_update_day'] = 0
df['ios_version'] = ''

for date_str, version in IOS_UPDATES:
    update_date = pd.to_datetime(date_str)
    mask = df['date'] == update_date
    df.loc[mask, 'is_ios_update_day'] = 1
    df.loc[mask, 'ios_version'] = version

# ВЕКТОРИЗОВАННЫЙ РАСЧЁТ days_after_ios_update
ios_dates_np = np.array([
    pd.Timestamp(d).to_datetime64().astype('datetime64[D]') 
    for d, _ in IOS_UPDATES
], dtype='datetime64[D]')

diff_ios = df_dates_np[:, None] - ios_dates_np[None, :]
diff_ios_f = diff_ios.astype(np.float64)
mask_ios = diff_ios_f > 0
diff_ios_masked = np.where(mask_ios, diff_ios_f, np.inf)

df['days_after_ios_update'] = np.min(diff_ios_masked, axis=1)
df['days_after_ios_update'] = np.where(
    np.isinf(df['days_after_ios_update']), 
    999, 
    df['days_after_ios_update']
)
df['days_after_ios_update'] = df['days_after_ios_update'].astype(int)

# Признак: неделя после обновления iOS
df['is_week_after_ios_update'] = (
    (df['days_after_ios_update'] >= 0) & 
    (df['days_after_ios_update'] <= 7)
).astype(int)

print(f"  + Признак is_ios_update_day: {df['is_ios_update_day'].sum():,} записей")
print(f"  + Признак is_week_after_ios_update: {df['is_week_after_ios_update'].sum():,} записей")

# =====================================================
# 4. КОМБИНИРОВАННЫЕ ПРИЗНАКИ
# =====================================================
df['is_freeze_during_update'] = (
    (df['is_freeze'] == 1) & 
    (df['days_after_ios_update'] <= 7)
).astype(int)

df['is_update_during_freeze'] = (
    (df['is_ios_update_day'] == 1) & 
    (df['is_freeze'] == 1)
).astype(int)

df['is_holiday_season'] = df['month'].isin([1, 12]).astype(int)

print(f"  + Комбинированные признаки добавлены")
print(f"  ⏱️  Время создания признаков событий: {time.time() - t1:.1f} сек")

# =====================================================
# 5. ВИЗУАЛИЗАЦИЯ ФРИЗОВ И ОБНОВЛЕНИЙ
# =====================================================
print("\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ ФРИЗОВ И ОБНОВЛЕНИЙ iOS")
print("="*60)

plots_path = DATA_PATH / "plots"
plots_path.mkdir(exist_ok=True)

fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Влияние фризов и обновлений iOS на позиции', fontsize=16, fontweight='bold')

# График 1: Средняя позиция с отметками фризов
ax = axes[0]
df_daily = df.groupby('date').agg({
    'position': lambda x: x[x != -1].mean(),
    'motiv': 'mean',
    'is_freeze': 'max',
    'is_ios_update_day': 'max'
}).reset_index()

ax.plot(df_daily['date'], df_daily['position'],
        color='steelblue', linewidth=1.5, label='Средняя позиция', alpha=0.7)

freeze_days = df_daily[df_daily['is_freeze'] == 1]['date']
for fd in freeze_days:
    ax.axvspan(fd, fd + pd.Timedelta(days=1), alpha=0.2, color='red', zorder=0)

update_days = df_daily[df_daily['is_ios_update_day'] == 1]['date']
for ud in update_days:
    ax.axvline(ud, color='green', linestyle='--', alpha=0.6, linewidth=1.5)

ax.set_title('Средняя позиция с отметками фризов (красный) и обновлений iOS (зелёный)')
ax.set_xlabel('Дата')
ax.set_ylabel('Средняя позиция')
ax.invert_yaxis()
ax.legend(['Средняя позиция', 'Фризы', 'iOS обновления'], loc='upper right')
ax.grid(True, alpha=0.3)

# График 2: Мотив с отметками
ax = axes[1]
ax.plot(df_daily['date'], df_daily['motiv'],
        color='purple', linewidth=1.5, label='Средний мотив', alpha=0.7)

for fd in freeze_days:
    ax.axvspan(fd, fd + pd.Timedelta(days=1), alpha=0.2, color='red', zorder=0)

for ud in update_days:
    ax.axvline(ud, color='green', linestyle='--', alpha=0.6, linewidth=1.5)

ax.set_title('Средний мотив с отметками фризов и обновлений')
ax.set_xlabel('Дата')
ax.set_ylabel('Средний мотив')
ax.legend(['Средний мотив', 'Фризы', 'iOS обновления'], loc='upper right')
ax.grid(True, alpha=0.3)

# График 3: Количество активных ключей
ax = axes[2]
df_active = df[df['motiv'] > 0].groupby('date')['keyword'].nunique().reset_index()
df_active.columns = ['date', 'active_keywords']

ax.plot(df_active['date'], df_active['active_keywords'],
        color='green', linewidth=1.5, label='Активных ключей', alpha=0.7)

for fd in freeze_days:
    ax.axvspan(fd, fd + pd.Timedelta(days=1), alpha=0.2, color='red', zorder=0)

for ud in update_days:
    ax.axvline(ud, color='green', linestyle='--', alpha=0.6, linewidth=1.5)

ax.set_title('Количество активных ключей')
ax.set_xlabel('Дата')
ax.set_ylabel('Активных ключей')
ax.legend(['Активные ключи', 'Фризы', 'iOS обновления'], loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(plots_path / 'freeze_and_updates.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График фризов и обновлений сохранён: {plots_path / 'freeze_and_updates.png'}")

# =====================================================
# 6. АНАЛИЗ ВЛИЯНИЯ
# =====================================================
print("\n" + "="*60)
print("АНАЛИЗ ВЛИЯНИЯ ФРИЗОВ И iOS ОБНОВЛЕНИЙ")
print("="*60)

df_freeze = df[df['is_freeze'] == 1]
df_no_freeze = df[df['is_freeze'] == 0]

print(f"\n  Периоды фризов:")
print(f"    - Записей: {len(df_freeze):,}")
print(f"    - Средняя позиция: {df_freeze[df_freeze['position'] != -1]['position'].mean():.2f}")
print(f"    - Средний мотив: {df_freeze['motiv'].mean():.2f}")
print(f"    - Среднее изменение позиции (1д): {df_freeze['position_delta_1d'].mean():.3f}")

print(f"\n  Вне фризов:")
print(f"    - Записей: {len(df_no_freeze):,}")
print(f"    - Средняя позиция: {df_no_freeze[df_no_freeze['position'] != -1]['position'].mean():.2f}")
print(f"    - Средний мотив: {df_no_freeze['motiv'].mean():.2f}")
print(f"    - Среднее изменение позиции (1д): {df_no_freeze['position_delta_1d'].mean():.3f}")

# Тест на различие
if len(df_freeze) > 20 and len(df_no_freeze) > 20:
    from scipy.stats import mannwhitneyu
    stat, p_value = mannwhitneyu(
        df_freeze['position_delta_1d'].dropna(),
        df_no_freeze['position_delta_1d'].dropna()
    )
    print(f"\n  Mann-Whitney U тест:")
    print(f"    - p-value: {p_value:.4f}")
    print(f"    - Результат: {'✅ Значимое различие' if p_value < 0.05 else '❌ Нет различий'}")

df_update_week = df[df['is_week_after_ios_update'] == 1]
df_no_update = df[df['is_week_after_ios_update'] == 0]

print(f"\n  Неделя после обновления iOS:")
print(f"    - Записей: {len(df_update_week):,}")
print(f"    - Средний мотив: {df_update_week['motiv'].mean():.2f}")
print(f"    - Δ позиции (1д): {df_update_week['position_delta_1d'].mean():.3f}")

print(f"\n  Обычные дни:")
print(f"    - Записей: {len(df_no_update):,}")
print(f"    - Средний мотив: {df_no_update['motiv'].mean():.2f}")
print(f"    - Δ позиции (1д): {df_no_update['position_delta_1d'].mean():.3f}")

# =====================================================
# 7. СОХРАНЕНИЕ ДАННЫХ
# =====================================================
print("\n" + "="*60)
print("СОХРАНЕНИЕ ДАННЫХ")
print("="*60)

df.to_csv(DATA_PATH / "clean_data.csv", index=False)
df.to_pickle(DATA_PATH / "clean_data.pkl")

# Сохраняем справочники
freeze_df = pd.DataFrame(FREEZE_PERIODS, columns=['start', 'end', 'name'])
freeze_df.to_csv(DATA_PATH / "freeze_periods.csv", index=False)

ios_df = pd.DataFrame(IOS_UPDATES, columns=['date', 'version'])
ios_df.to_csv(DATA_PATH / "ios_updates.csv", index=False)

print(f"\n✅ Данные сохранены:")
print(f"  - CSV: {DATA_PATH / 'clean_data.csv'}")
print(f"  - Pickle: {DATA_PATH / 'clean_data.pkl'}")
print(f"  - Справочник фризов: {DATA_PATH / 'freeze_periods.csv'}")
print(f"  - Справочник iOS: {DATA_PATH / 'ios_updates.csv'}")

# =====================================================
# 8. ИТОГОВАЯ СТАТИСТИКА
# =====================================================
elapsed = time.time() - start_time

print(f"\n" + "="*60)
print("ИТОГОВАЯ СТАТИСТИКА ШАГА 1")
print("="*60)
print(f"  - Всего записей: {len(df):,}")
print(f"  - Уникальных ключей: {df['keyword'].nunique()}")
print(f"  - Всего признаков: {len(df.columns)}")
print(f"  - Период: {df['date'].min()} - {df['date'].max()}")
print(f"  - Записей в фризах: {df['is_freeze'].sum():,}")
print(f"  - Дней с iOS обновлениями: {df['is_ios_update_day'].sum()}")
print(f"  - Записей в неделю после iOS: {df['is_week_after_ios_update'].sum():,}")
print(f"  - Время выполнения: {elapsed:.1f} сек ({elapsed/60:.1f} мин)")

print("\n" + "="*80)
print("✅ ШАГ 1 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 1: ЗАГРУЗКА, ПРЕДОБРАБОТКА И ДЕДУПЛИКАЦИЯ ДАННЫХ

Парсинг файла: Sound Amplifier_2024.xlsx
  - Найдено дат: 472
  - Найдено органики US: 424
  - Найдено активаций US: 403
  - Диапазон ключевых слов: 4 - 130
  - Загружено 42,416 записей
  - Уникальных ключей: 122
  - Период: 2023-12-28 00:00:00 - 2025-04-07 00:00:00

Парсинг файла: Sound Amplifier_2025.xlsx
  - Найдено дат: 416
  - Найдено органики US: 416
  - Найдено активаций US: 395
  - Диапазон ключевых слов: 4 - 130
  - Загружено 45,539 записей
  - Уникальных ключей: 123
  - Период: 2025-01-01 00:00:00 - 2026-02-10 00:00:00

Парсинг файла: Sound Amplifier_2026.xlsx
  - Найдено дат: 196
  - Найдено органики US: 172
  - Найдено активаций US: 172
  - Диапазон ключевых слов: 4 - 121
  - Загружено 19,773 записей
  - Уникальных ключей: 117
  - Период: 2026-01-01 00:00:00 - 2026-07-15 00:00:00

ДО ДЕДУПЛИКАЦИИ:
  - Всего записей: 107,728
  - Уникальных ключей: 134
  - Период: 2023-12-28 00:00:00 - 2026-07-15 00:00:00

ДЕДУПЛИКАЦИЯ ДА

# Шаг 2. РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)

In [3]:
# =====================================================
# ШАГ 2: РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)
# =====================================================
print("\n" + "="*80)
print("ШАГ 2: РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)")
print("="*80)

df = pd.read_pickle(DATA_PATH / "clean_data.pkl")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")
print(f"Период: {df['date'].min()} - {df['date'].max()}")

# =====================================================
# 1. ОБЩАЯ СТАТИСТИКА
# =====================================================
print("\n" + "="*60)
print("1. ОБЩАЯ СТАТИСТИКА")
print("="*60)

print("\nСтатистика по числовым колонкам:")
print(df[['motiv', 'position', 'organic_us', 'activation_us']].describe())

print("\nУникальные значения по ключевым колонкам:")
print(f"  - Количество ключевых слов: {df['keyword'].nunique()}")
print(f"  - Количество дат: {df['date'].nunique()}")
print(f"  - Количество лет: {df['year'].nunique()}")
print(f"  - Всего признаков: {len(df.columns)}")

# =====================================================
# 2. РАСПРЕДЕЛЕНИЕ МОТИВА И ПОЗИЦИЙ
# =====================================================
print("\n" + "="*60)
print("2. РАСПРЕДЕЛЕНИЕ МОТИВА И ПОЗИЦИЙ")
print("="*60)

print("\nСтатистика по мотиву:")
motiv_pos = df[df['motiv'] > 0]['motiv']
print(f"  - Записей с мотивом > 0: {len(motiv_pos)} ({len(motiv_pos)/len(df)*100:.1f}%)")
print(f"  - Записей без мотива: {(df['motiv'] == 0).sum()} ({(df['motiv'] == 0).sum()/len(df)*100:.1f}%)")
print(f"  - Средний мотив (все записи): {df['motiv'].mean():.2f}")
print(f"  - Средний мотив (только > 0): {motiv_pos.mean():.2f}")
print(f"  - Медианный мотив (только > 0): {motiv_pos.median():.2f}")
print(f"  - Максимальный мотив: {df['motiv'].max():.0f}")

print("\nСтатистика по позициям:")
valid_pos_mask = df['position'] != -1
position_valid_df = df[valid_pos_mask]
position_values = position_valid_df['position']

print(f"  - Записей с позицией: {len(position_valid_df)} ({len(position_valid_df)/len(df)*100:.1f}%)")
print(f"  - Записей без позиции: {(df['position'] == -1).sum()} ({(df['position'] == -1).sum()/len(df)*100:.1f}%)")
if len(position_values) > 0:
    print(f"  - Средняя позиция: {position_values.mean():.2f}")
    print(f"  - Медианная позиция: {position_values.median():.2f}")
    print(f"  - Минимальная позиция: {position_values.min():.0f}")
    print(f"  - Максимальная позиция: {position_values.max():.0f}")

# =====================================================
# 3. ТОП-20 КЛЮЧЕВЫХ СЛОВ
# =====================================================
print("\n" + "="*60)
print("3. ТОП-20 КЛЮЧЕВЫХ СЛОВ")
print("="*60)

top20 = df['keyword'].value_counts().head(20)
print("\nТоп-20 по количеству записей:")
for i, (keyword, count) in enumerate(top20.items(), 1):
    print(f"  {i:2d}. {keyword:35s} - {count:5d} записей")

top20_keywords = top20.index.tolist()
df_top20 = df[df['keyword'].isin(top20_keywords)]

df_top20_unique = df_top20.groupby(['keyword', 'date']).agg({
    'motiv': 'mean',
    'position': 'first',
    'organic_us': 'mean',
    'activation_us': 'mean',
    'motiv_scenario': 'first',
    'position_delta_1d': 'first'
}).reset_index()

print("\nСтатистика по топ-20 ключам:")
stats_top20 = df_top20_unique.groupby('keyword').agg({
    'motiv': ['mean', 'max', 'sum'],
    'position': ['mean', 'min', 'max']
}).round(2)
stats_top20.columns = ['motiv_mean', 'motiv_max', 'motiv_sum', 'position_mean', 'position_min', 'position_max']
stats_top20 = stats_top20.sort_values('motiv_sum', ascending=False)

print("\n  {:<30s} {:>12s} {:>12s} {:>12s} {:>12s} {:>12s} {:>12s}".format(
    'Ключ', 'Мотив_сред', 'Мотив_макс', 'Мотив_сумма', 'Позиция_сред', 'Позиция_мин', 'Позиция_макс'
))
print("  " + "-"*100)
for keyword, row in stats_top20.head(20).iterrows():
    print("  {:<30s} {:>12.2f} {:>12.0f} {:>12.0f} {:>12.2f} {:>12.0f} {:>12.0f}".format(
        keyword[:30], 
        row['motiv_mean'], 
        row['motiv_max'], 
        row['motiv_sum'],
        row['position_mean'],
        row['position_min'],
        row['position_max']
    ))

# =====================================================
# 4. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ
# =====================================================
print("\n" + "="*60)
print("4. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ")
print("="*60)

print("\nКорреляция мотива и позиции:")
keyword_corr = df_top20_unique.groupby('keyword').apply(
    lambda x: x[['motiv', 'position']].corr().iloc[0, 1] if len(x) > 1 else np.nan
).dropna()

if len(keyword_corr) > 0:
    print(f"  - Средняя корреляция: {keyword_corr.mean():.3f}")
    print(f"  - Медианная корреляция: {keyword_corr.median():.3f}")
    print(f"  - Максимальная корреляция: {keyword_corr.max():.3f}")
    print(f"  - Минимальная корреляция: {keyword_corr.min():.3f}")

print("\nКорреляция между ключевыми словами (на основе позиций):")
pivot_position = df_top20_unique.pivot_table(
    index='date', 
    columns='keyword', 
    values='position'
).dropna(axis=1, how='all')

avg_corr = 0
corr_values = pd.Series()

if pivot_position.shape[1] > 1:
    corr_matrix = pivot_position.corr()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    corr_values = upper_tri.stack()
    corr_values = corr_values[~np.isnan(corr_values)]
    
    if len(corr_values) > 0:
        avg_corr = corr_values.mean()
        print(f"  - Средняя корреляция между ключами: {avg_corr:.3f}")
        print(f"  - Максимальная корреляция: {corr_values.max():.3f}")
        print(f"  - Минимальная корреляция: {corr_values.min():.3f}")
        
        print("\n  Топ-5 пар ключей с максимальной корреляцией:")
        corr_pairs = corr_values.sort_values(ascending=False).head(5)
        for (k1, k2), corr in corr_pairs.items():
            print(f"    - {k1} ↔ {k2}: {corr:.3f}")

# =====================================================
# 5. СЦЕНАРИИ ИЗМЕНЕНИЯ МОТИВА
# =====================================================
print("\n" + "="*60)
print("5. СЦЕНАРИИ ИЗМЕНЕНИЯ МОТИВА")
print("="*60)

scenario_names = {
    'stable': 'стабильный',
    'sharp_growth': 'резкий рост',
    'sharp_decline': 'резкий спад',
    'gentle_growth': 'плавный рост',
    'gentle_decline': 'плавный спад'
}

if 'motiv_scenario' in df.columns:
    scenario_counts = df['motiv_scenario'].value_counts()
    print("\nРаспределение сценариев:")
    for scenario, count in scenario_counts.items():
        name = scenario_names.get(scenario, scenario)
        print(f"  - {name}: {count} записей ({count/len(df)*100:.1f}%)")

print("\nЭффективность сценариев (изменение позиции через 1 день):")
if 'position_delta_1d' in df_top20_unique.columns and 'motiv_scenario' in df_top20_unique.columns:
    df_scenario_effect = df_top20_unique.dropna(subset=['position_delta_1d', 'motiv_scenario'])
    
    if len(df_scenario_effect) > 0:
        scenario_effect = df_scenario_effect.groupby('motiv_scenario').agg({
            'position_delta_1d': ['mean', 'std', 'count']
        }).round(2)
        scenario_effect.columns = ['mean_delta', 'std_delta', 'count']
        
        print("\n  {:<20s} {:>12s} {:>12s} {:>10s}".format(
            'Сценарий', 'Среднее Δ', 'Стд. откл.', 'Кол-во'
        ))
        print("  " + "-"*60)
        for scenario, row in scenario_effect.iterrows():
            name = scenario_names.get(scenario, scenario)
            print("  {:<20s} {:>12.2f} {:>12.2f} {:>10.0f}".format(
                name, row['mean_delta'], row['std_delta'], row['count']
            ))

# =====================================================
# 6. ВИЗУАЛИЗАЦИЯ
# =====================================================
print("\n" + "="*60)
print("6. ВИЗУАЛИЗАЦИЯ")
print("="*60)

plots_path = DATA_PATH / "plots"
plots_path.mkdir(exist_ok=True)

# 6.1. Распределение мотива и позиций
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Анализ мотива и позиций', fontsize=16, fontweight='bold')

ax = axes[0, 0]
motiv_positive = df[df['motiv'] > 0]['motiv']
motiv_positive.hist(bins=30, edgecolor='black', alpha=0.7, color='steelblue', ax=ax)
ax.set_title('Распределение мотива (только > 0)')
ax.set_xlabel('Мотив')
ax.set_ylabel('Частота')
motiv_mean = motiv_positive.mean()
ax.axvline(motiv_mean, color='red', linestyle='--', label=f'Средний: {motiv_mean:.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
position_valid_vals = df[df['position'] != -1]['position']
position_valid_vals.hist(bins=50, edgecolor='black', alpha=0.7, color='orange', ax=ax)
ax.set_title('Распределение позиций')
ax.set_xlabel('Позиция')
ax.set_ylabel('Частота')
if len(position_valid_vals) > 0:
    pos_mean = position_valid_vals.mean()
    pos_median = position_valid_vals.median()
    ax.axvline(pos_mean, color='red', linestyle='--', label=f'Средняя: {pos_mean:.2f}')
    ax.axvline(pos_median, color='blue', linestyle='--', label=f'Медиана: {pos_median:.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
top20.plot(kind='barh', ax=ax, color='green')
ax.set_title('Топ-20 ключевых слов по количеству записей')
ax.set_xlabel('Количество записей')
ax.set_ylabel('Ключевое слово')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
if len(stats_top20) > 0:
    ax.scatter(stats_top20['motiv_mean'], stats_top20['position_mean'], 
               alpha=0.6, s=80, color='purple')
    ax.set_title('Средний мотив vs Средняя позиция (топ-20)')
    ax.set_xlabel('Средний мотив')
    ax.set_ylabel('Средняя позиция')
    ax.grid(True, alpha=0.3)
    
    for keyword in stats_top20.head(10).index:
        ax.annotate(keyword[:15], 
                    (stats_top20.loc[keyword, 'motiv_mean'], 
                     stats_top20.loc[keyword, 'position_mean']),
                    fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig(plots_path / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График сохранен: {plots_path / 'eda_overview.png'}")

# 6.2. Тепловая карта корреляций
if 'corr_matrix' in locals() and corr_matrix.shape[0] > 1:
    fig, ax = plt.subplots(figsize=(14, 12))
    
    if corr_matrix.shape[0] > 15:
        top_keys = top20.head(15).index.tolist()
        corr_subset = corr_matrix.loc[top_keys, top_keys]
    else:
        corr_subset = corr_matrix
    
    mask = np.triu(np.ones_like(corr_subset, dtype=bool))
    sns.heatmap(corr_subset, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
                center=0, square=True, ax=ax, cbar_kws={'label': 'Корреляция'})
    ax.set_title('Корреляция позиций между ключевыми словами', fontsize=14)
    plt.tight_layout()
    plt.savefig(plots_path / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  + Тепловая карта сохранена: {plots_path / 'correlation_heatmap.png'}")

# 6.3. Временные ряды (топ-5 ключей)
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

top5 = top20.head(5).index.tolist()
df_top5 = df[df['keyword'].isin(top5)]

ax = axes[0]
for keyword in top5:
    keyword_data = df_top5[df_top5['keyword'] == keyword]
    ax.plot(keyword_data['date'], keyword_data['motiv'], label=keyword, alpha=0.7)
ax.set_title('Динамика мотива (топ-5 ключей)')
ax.set_xlabel('Дата')
ax.set_ylabel('Мотив')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for keyword in top5:
    keyword_data = df_top5[df_top5['keyword'] == keyword]
    keyword_data = keyword_data[keyword_data['position'] != -1]
    if len(keyword_data) > 0:
        ax.plot(keyword_data['date'], keyword_data['position'], label=keyword, alpha=0.7)
ax.set_title('Динамика позиций (топ-5 ключей)')
ax.set_xlabel('Дата')
ax.set_ylabel('Позиция')
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(plots_path / 'timeseries_top5.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + Временные ряды сохранены: {plots_path / 'timeseries_top5.png'}")

# 6.4. Ежемесячный анализ
print("\n  + Построение ежемесячного анализа...")

df_monthly = df.groupby(df['date'].dt.to_period('M')).agg({
    'motiv': ['mean', 'sum', 'count'],
    'position': 'mean'
}).reset_index()
df_monthly.columns = ['period', 'motiv_mean', 'motiv_sum', 'motiv_count', 'position_mean']
df_monthly['period_str'] = df_monthly['period'].astype(str)

monthly_days_data = []
for period, group in df.groupby(df['date'].dt.to_period('M')):
    total_days = group['date'].nunique()
    days_with_motiv = group[group['motiv'] > 0]['date'].nunique()
    monthly_days_data.append({
        'period': period,
        'total_days': total_days,
        'days_with_motiv': days_with_motiv
    })

monthly_days = pd.DataFrame(monthly_days_data)
monthly_days['period_str'] = monthly_days['period'].astype(str)
monthly_days['days_ratio'] = (monthly_days['days_with_motiv'] / monthly_days['total_days']) * 100

monthly_active_data = []
for period, group in df[df['motiv'] > 0].groupby(df['date'].dt.to_period('M')):
    active_keywords = group['keyword'].nunique()
    monthly_active_data.append({
        'period': period,
        'active_keywords': active_keywords
    })

monthly_active = pd.DataFrame(monthly_active_data)
monthly_active['period_str'] = monthly_active['period'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Ежемесячный анализ мотива и позиций', fontsize=16, fontweight='bold')

ax1 = axes[0, 0]
ax2 = ax1.twinx()

ax1.bar(df_monthly['period_str'], df_monthly['motiv_mean'], 
        alpha=0.7, color='steelblue', label='Средний мотив')
ax1.set_xlabel('Месяц')
ax1.set_ylabel('Средний мотив', color='steelblue')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

ax2.plot(df_monthly['period_str'], df_monthly['position_mean'], 
         color='red', marker='o', linewidth=2, label='Средняя позиция')
ax2.set_ylabel('Средняя позиция', color='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Средний мотив и средняя позиция по месяцам')

ax = axes[0, 1]
ax.bar(df_monthly['period_str'], df_monthly['motiv_sum'], 
       alpha=0.7, color='green', label='Суммарный мотив')
ax.set_xlabel('Месяц')
ax.set_ylabel('Суммарный мотив')
ax.set_title('Суммарный мотив по месяцам')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

max_sum = df_monthly['motiv_sum'].max() if df_monthly['motiv_sum'].max() > 0 else 1
for i, v in enumerate(df_monthly['motiv_sum']):
    ax.text(i, v + max_sum * 0.02, f'{int(v)}', ha='center', va='bottom', fontsize=8)

ax = axes[1, 0]
ax.bar(monthly_days['period_str'], monthly_days['days_ratio'], 
       alpha=0.7, color='coral')
ax.set_xlabel('Месяц')
ax.set_ylabel('Доля дней с мотивом > 0 (%)')
ax.set_title('Доля дней с мотивом > 0 по месяцам')
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=30, color='red', linestyle='--', linewidth=2, label='30% порог')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

for i, v in enumerate(monthly_days['days_ratio']):
    ax.text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

ax = axes[1, 1]
ax.bar(monthly_active['period_str'], monthly_active['active_keywords'], 
       alpha=0.7, color='purple')
ax.set_xlabel('Месяц')
ax.set_ylabel('Количество активных ключей')
ax.set_title('Количество ключей с мотивом > 0 по месяцам')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

for i, v in enumerate(monthly_active['active_keywords']):
    ax.text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(plots_path / 'monthly_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + Ежемесячный анализ сохранен: {plots_path / 'monthly_analysis.png'}")

print("\n" + "="*80)
print("✅ ШАГ 2 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 2: РАЗВЕДОЧНЫЙ АНАЛИЗ ДАННЫХ (EDA)

Размер данных: 92,955 записей
Уникальных ключей: 134
Период: 2023-12-28 00:00:00 - 2026-07-15 00:00:00

1. ОБЩАЯ СТАТИСТИКА

Статистика по числовым колонкам:
              motiv      position    organic_us  activation_us
count  92955.000000  92955.000000  92955.000000   92955.000000
mean       0.765037     45.849701     66.070615      67.668754
std        2.920837     48.225968     58.254404      55.400847
min        0.000000     -1.000000   -167.000000    -167.000000
25%        0.000000      8.000000     49.000000      49.000000
50%        0.000000     28.000000     73.000000      73.000000
75%        0.000000     69.000000     89.000000      89.000000
max      200.000000    249.000000    424.000000     424.000000

Уникальные значения по ключевым колонкам:
  - Количество ключевых слов: 134
  - Количество дат: 930
  - Количество лет: 3
  - Всего признаков: 121

2. РАСПРЕДЕЛЕНИЕ МОТИВА И ПОЗИЦИЙ

Статистика по мотиву:
  - Записей с мотивом > 0: 1

# 3 Расчет требуемого мотива

In [4]:
# =====================================================
# ШАГ 3: РАСЧЁТ ТРЕБУЕМОГО МОТИВА ДЛЯ ЦЕЛЕВОЙ ПОЗИЦИИ
# =====================================================
print("\n" + "="*80)
print("ШАГ 3: РАСЧЁТ ТРЕБУЕМОГО МОТИВА ДЛЯ ЦЕЛЕВОЙ ПОЗИЦИИ")
print("="*80)

df = pd.read_pickle(DATA_PATH / "clean_data.pkl")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")

# =====================================================
# 1. АГРЕГАЦИЯ ДАННЫХ ПО КЛЮЧАМ
# =====================================================
print("\n" + "="*60)
print("1. АГРЕГАЦИЯ ДАННЫХ ПО КЛЮЧАМ")
print("="*60)

keyword_stats = df.groupby('keyword').agg({
    'position': ['mean', 'min', 'max', 'std', 'count'],
    'motiv': ['mean', 'sum', 'max', 'std', 'count'],
    'organic_us': ['mean', 'sum'],
    'activation_us': ['mean', 'sum']
}).round(3)

keyword_stats.columns = [
    'pos_mean', 'pos_min', 'pos_max', 'pos_std', 'pos_count',
    'motiv_mean', 'motiv_sum', 'motiv_max', 'motiv_std', 'motiv_count',
    'organic_mean', 'organic_sum',
    'activation_mean', 'activation_sum'
]
keyword_stats = keyword_stats.reset_index()

print(f"  - Всего ключей: {len(keyword_stats)}")
print(f"  - Ключей с позицией: {(keyword_stats['pos_mean'] != -1).sum()}")
print(f"  - Ключей с мотивом > 0: {(keyword_stats['motiv_sum'] > 0).sum()}")

# =====================================================
# 2. РАСЧЁТ ЭФФЕКТИВНОСТИ МОТИВА ПО КЛЮЧАМ
# =====================================================
print("\n" + "="*60)
print("2. РАСЧЁТ ЭФФЕКТИВНОСТИ МОТИВА ПО КЛЮЧАМ")
print("="*60)

def calculate_motiv_efficiency(df, keyword):
    """
    Рассчитывает эффективность мотива для ключевого слова:
    - Сколько позиций даёт 1 единица мотива
    - Какой мотив нужен для достижения целевой позиции
    """
    kw_data = df[df['keyword'] == keyword].sort_values('date')
    
    if len(kw_data) < 10:
        return None
    
    # Данные с мотивом > 0
    with_motiv = kw_data[kw_data['motiv'] > 0]
    without_motiv = kw_data[kw_data['motiv'] == 0]
    
    if len(with_motiv) < 5:
        return None
    
    # Средняя позиция с мотивом и без
    avg_pos_with = with_motiv['position'].mean()
    avg_pos_without = without_motiv['position'].mean() if len(without_motiv) > 0 else avg_pos_with
    
    # Средний мотив
    avg_motiv = with_motiv['motiv'].mean()
    
    # Эффективность: сколько позиций даёт 1 единица мотива
    if avg_motiv > 0 and avg_pos_without != -1:
        efficiency = (avg_pos_without - avg_pos_with) / avg_motiv
    else:
        efficiency = 0
    
    # Минимальная позиция (лучшая)
    best_pos = kw_data[kw_data['position'] != -1]['position'].min()
    best_motiv_rows = kw_data[kw_data['position'] == best_pos]
    best_motiv = best_motiv_rows['motiv'].mean() if len(best_motiv_rows) > 0 else 0
    
    # Корреляция мотива и позиции
    valid = kw_data[(kw_data['position'] != -1) & (kw_data['motiv'] > 0)]
    if len(valid) > 5:
        corr = valid[['motiv', 'position']].corr().iloc[0, 1]
    else:
        corr = 0
    
    return {
        'keyword': keyword,
        'avg_pos_with_motiv': round(avg_pos_with, 2),
        'avg_pos_without_motiv': round(avg_pos_without, 2),
        'avg_motiv': round(avg_motiv, 2),
        'max_motiv': float(with_motiv['motiv'].max()),
        'efficiency': round(efficiency, 4),
        'best_position': float(best_pos),
        'best_motiv': round(best_motiv, 2),
        'correlation': round(corr, 4),
        'n_with_motiv': len(with_motiv),
        'n_without_motiv': len(without_motiv),
        'n_total': len(kw_data)
    }

# Расчёт эффективности для всех ключей
print("\nРасчёт эффективности мотива для всех ключей...")
motiv_efficiency = []
for keyword in df['keyword'].unique():
    result = calculate_motiv_efficiency(df, keyword)
    if result:
        motiv_efficiency.append(result)

motiv_efficiency_df = pd.DataFrame(motiv_efficiency)
print(f"  - Рассчитано для {len(motiv_efficiency_df)} ключей")

# Топ-10 самых эффективных ключей
print("\nТоп-10 самых эффективных ключей:")
if len(motiv_efficiency_df) > 0:
    top_efficient = motiv_efficiency_df.sort_values('efficiency', ascending=False).head(10)
    print("  {:<35s} {:>15s} {:>12s} {:>12s}".format(
        'Ключ', 'Эффективность', 'Ср.мотив', 'Корреляция'
    ))
    print("  " + "-"*80)
    for _, row in top_efficient.iterrows():
        print("  {:<35s} {:>15.3f} {:>12.2f} {:>12.3f}".format(
            row['keyword'][:35],
            row['efficiency'],
            row['avg_motiv'],
            row['correlation']
        ))

# =====================================================
# 3. ФУНКЦИЯ РАСЧЁТА ТРЕБУЕМОГО МОТИВА
# =====================================================
print("\n" + "="*60)
print("3. ФУНКЦИЯ РАСЧЁТА ТРЕБУЕМОГО МОТИВА")
print("="*60)

def calculate_required_motiv(keyword, target_position, df, motiv_efficiency_df):
    """
    Рассчитывает требуемый мотив для достижения целевой позиции
    
    Args:
        keyword: ключевое слово
        target_position: целевая позиция (1-250)
        df: DataFrame с данными
        motiv_efficiency_df: DataFrame с эффективностью мотива
    
    Returns:
        dict с требуемым мотивом и прогнозом
    """
    kw_data = df[df['keyword'] == keyword]
    
    if len(kw_data) == 0:
        return {'error': f'Ключ "{keyword}" не найден'}
    
    # Текущая позиция
    valid_pos = kw_data[kw_data['position'] != -1]
    current_pos = valid_pos['position'].iloc[-1] if len(valid_pos) > 0 else -1
    current_motiv = kw_data['motiv'].iloc[-1] if len(kw_data) > 0 else 0
    
    # Эффективность мотива
    eff_row = motiv_efficiency_df[motiv_efficiency_df['keyword'] == keyword]
    
    if len(eff_row) == 0:
        efficiency = 1.0
        avg_motiv = kw_data['motiv'].mean()
        max_motiv = kw_data['motiv'].max()
        correlation = 0
    else:
        eff = eff_row.iloc[0]
        efficiency = eff['efficiency']
        avg_motiv = eff['avg_motiv']
        max_motiv = eff['max_motiv']
        correlation = eff['correlation']
    
    # Разница между текущей и целевой позицией
    position_diff = current_pos - target_position
    
    if position_diff <= 0:
        return {
            'keyword': keyword,
            'current_position': round(current_pos, 1),
            'target_position': target_position,
            'position_diff': 0,
            'current_motiv': round(current_motiv, 2),
            'required_motiv': 0,
            'required_motiv_int': 0,
            'efficiency': round(efficiency, 3),
            'correlation': round(correlation, 3),
            'predicted_position_1d': round(current_pos, 1),
            'predicted_position_7d': round(current_pos, 1),
            'confidence': 'high',
            'n_records': len(kw_data),
            'message': f'✅ Уже на целевой позиции ({current_pos:.1f} ≤ {target_position})',
            'recommendation': 'Поддерживать текущий мотив'
        }
    
    # Расчёт требуемого мотива
    if efficiency > 0.1:
        required_motiv = position_diff / efficiency
    else:
        required_motiv = position_diff * 0.5
    
    required_motiv = min(max(required_motiv, 1), 30)
    
    # Прогноз на 1 день
    if efficiency > 0.1:
        predicted_pos_1d = current_pos - (required_motiv * efficiency)
    else:
        predicted_pos_1d = current_pos - required_motiv * 0.5
    
    # Прогноз через 7 дней
    if efficiency > 0.1:
        predicted_pos_7d = current_pos - (required_motiv * efficiency * 7 * 0.3)
    else:
        predicted_pos_7d = current_pos - required_motiv * 0.5 * 7 * 0.3
    
    # Уверенность
    if len(kw_data) > 100 and efficiency > 0.5 and abs(correlation) > 0.3:
        confidence = 'high'
    elif len(kw_data) > 50 and efficiency > 0.2:
        confidence = 'medium'
    else:
        confidence = 'low'
    
    # Рекомендация
    if required_motiv <= 2:
        recommendation = 'Небольшое увеличение мотива (по правилу R5)'
    elif required_motiv <= 5:
        recommendation = 'Умеренное увеличение мотива (по правилу R9)'
    elif required_motiv <= 10:
        recommendation = 'Значительное увеличение мотива (по правилу R8)'
    else:
        recommendation = 'Агрессивное продвижение (по правилу R12)'
    
    return {
        'keyword': keyword,
        'current_position': round(current_pos, 1),
        'target_position': target_position,
        'position_diff': round(position_diff, 1),
        'current_motiv': round(current_motiv, 2),
        'required_motiv': round(required_motiv, 2),
        'required_motiv_int': int(np.ceil(required_motiv)),
        'efficiency': round(efficiency, 3),
        'correlation': round(correlation, 3),
        'predicted_position_1d': round(predicted_pos_1d, 1),
        'predicted_position_7d': round(predicted_pos_7d, 1),
        'confidence': confidence,
        'n_records': len(kw_data),
        'message': f'Для достижения позиции {target_position} требуется мотив ≈ {int(np.ceil(required_motiv))}',
        'recommendation': recommendation
    }

# Тестовый расчёт для топ-10 ключей
print("\nПримеры расчёта требуемого мотива (цель: топ-10):")
print("  {:<30s} {:>10s} {:>10s} {:>12s} {:>12s} {:>10s}".format(
    'Ключ', 'Текущая', 'Целевая', 'Треб.мотив', 'Прогноз 1д', 'Уверен.'
))
print("  " + "-"*90)

top10 = df['keyword'].value_counts().head(10).index.tolist()
for keyword in top10:
    result = calculate_required_motiv(keyword, 10, df, motiv_efficiency_df)
    if 'error' not in result:
        print("  {:<30s} {:>10.1f} {:>10d} {:>12.2f} {:>12.1f} {:>10s}".format(
            keyword[:30],
            result['current_position'],
            result['target_position'],
            result['required_motiv'],
            result['predicted_position_1d'],
            result['confidence']
        ))

# =====================================================
# 4. ВИЗУАЛИЗАЦИЯ
# =====================================================
print("\n" + "="*60)
print("4. ВИЗУАЛИЗАЦИЯ")
print("="*60)

plots_path = DATA_PATH / "plots"

if len(motiv_efficiency_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Эффективность мотива по ключевым словам', fontsize=16, fontweight='bold')
    
    # 1. Топ-20 по эффективности
    ax = axes[0, 0]
    top20_eff = motiv_efficiency_df.sort_values('efficiency', ascending=False).head(20)
    colors = ['green' if e > 1 else 'orange' if e > 0.3 else 'red' for e in top20_eff['efficiency']]
    ax.barh(top20_eff['keyword'], top20_eff['efficiency'], color=colors, alpha=0.7)
    ax.set_xlabel('Эффективность (позиций за 1 ед. мотива)')
    ax.set_title('Топ-20 ключей по эффективности мотива')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    # 2. Корреляция эффективности и среднего мотива
    ax = axes[0, 1]
    ax.scatter(motiv_efficiency_df['avg_motiv'], motiv_efficiency_df['efficiency'], 
               alpha=0.6, s=60, c=motiv_efficiency_df['correlation'], cmap='RdYlGn')
    ax.set_xlabel('Средний мотив')
    ax.set_ylabel('Эффективность')
    ax.set_title('Средний мотив vs Эффективность')
    ax.grid(True, alpha=0.3)
    
    # 3. Распределение эффективности
    ax = axes[1, 0]
    motiv_efficiency_df['efficiency'].hist(bins=30, edgecolor='black', alpha=0.7, color='steelblue', ax=ax)
    ax.axvline(motiv_efficiency_df['efficiency'].mean(), color='red', linestyle='--', 
               label=f'Средняя: {motiv_efficiency_df["efficiency"].mean():.3f}')
    ax.axvline(motiv_efficiency_df['efficiency'].median(), color='blue', linestyle='--',
               label=f'Медиана: {motiv_efficiency_df["efficiency"].median():.3f}')
    ax.set_xlabel('Эффективность')
    ax.set_ylabel('Количество ключей')
    ax.set_title('Распределение эффективности мотива')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. Корреляция мотива и позиции по ключам
    ax = axes[1, 1]
    ax.scatter(motiv_efficiency_df['correlation'], motiv_efficiency_df['efficiency'],
               alpha=0.6, s=60, color='purple')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Корреляция мотива и позиции')
    ax.set_ylabel('Эффективность')
    ax.set_title('Корреляция vs Эффективность')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(plots_path / 'motiv_efficiency.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  + График эффективности сохранен: {plots_path / 'motiv_efficiency.png'}")

# =====================================================
# 5. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("5. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

motiv_efficiency_df.to_csv(DATA_PATH / "motiv_efficiency.csv", index=False)
print(f"  + Эффективность мотива сохранена: {DATA_PATH / 'motiv_efficiency.csv'}")

keyword_stats.to_csv(DATA_PATH / "keyword_stats.csv", index=False)
print(f"  + Статистика по ключам сохранена: {DATA_PATH / 'keyword_stats.csv'}")

# =====================================================
# 6. КЛЮЧЕВЫЕ ВЫВОДЫ
# =====================================================
print("\n" + "="*60)
print("6. КЛЮЧЕВЫЕ ВЫВОДЫ")
print("="*60)

if len(motiv_efficiency_df) > 0:
    best_eff = motiv_efficiency_df.loc[motiv_efficiency_df['efficiency'].idxmax()]
    worst_eff = motiv_efficiency_df.loc[motiv_efficiency_df['efficiency'].idxmin()]
    
    print(f"""
📊 РЕЗУЛЬТАТЫ АНАЛИЗА ЭФФЕКТИВНОСТИ МОТИВА:

1. ОБЩАЯ СТАТИСТИКА:
   - Всего ключей проанализировано: {len(motiv_efficiency_df)}
   - Средняя эффективность: {motiv_efficiency_df['efficiency'].mean():.3f}
   - Медианная эффективность: {motiv_efficiency_df['efficiency'].median():.3f}

2. ЛУЧШИЙ КЛЮЧ:
   - {best_eff['keyword']}
   - Эффективность: {best_eff['efficiency']:.3f} позиций за 1 ед. мотива
   - Средний мотив: {best_eff['avg_motiv']:.2f}
   
3. САМЫЙ СЛАБЫЙ КЛЮЧ:
   - {worst_eff['keyword']}
   - Эффективность: {worst_eff['efficiency']:.3f}
   - Средний мотив: {worst_eff['avg_motiv']:.2f}

4. ИНТЕРПРЕТАЦИЯ ЭФФЕКТИВНОСТИ:
   - Эффективность > 1.0 → 1 единица мотива даёт больше 1 позиции (отлично)
   - Эффективность 0.3-1.0 → умеренная эффективность
   - Эффективность < 0.3 → низкая эффективность, требуется больше мотива

5. РЕКОМЕНДАЦИИ:
   - Для ключей с высокой эффективностью используйте умеренный мотив
   - Для ключей с низкой эффективностью требуется агрессивное продвижение
   - Учитывайте правило R12 (мотив ≤ 30% от трафика)
""")

print("\n" + "="*80)
print("✅ ШАГ 3 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 3: РАСЧЁТ ТРЕБУЕМОГО МОТИВА ДЛЯ ЦЕЛЕВОЙ ПОЗИЦИИ

Размер данных: 92,955 записей
Уникальных ключей: 134

1. АГРЕГАЦИЯ ДАННЫХ ПО КЛЮЧАМ
  - Всего ключей: 134
  - Ключей с позицией: 132
  - Ключей с мотивом > 0: 107

2. РАСЧЁТ ЭФФЕКТИВНОСТИ МОТИВА ПО КЛЮЧАМ

Расчёт эффективности мотива для всех ключей...
  - Рассчитано для 103 ключей

Топ-10 самых эффективных ключей:
  Ключ                                  Эффективность     Ср.мотив   Корреляция
  --------------------------------------------------------------------------------
  bass app                                     60.380         2.27       -0.371
  listening                                    54.194         1.90       -0.468
  bass boost                                   48.495         1.30       -0.470
  increase volume                              45.517         1.33       -0.246
  free music booster                           40.607         1.44       -0.344
  louder volume booster                        33.816         1.66

# 3.2 ML

In [5]:
# =====================================================
# ШАГ 4: ML МОДЕЛИРОВАНИЕ С РАСШИРЕННЫМИ ПРИЗНАКАМИ
# =====================================================
print("\n" + "="*80)
print("ШАГ 4: ML МОДЕЛИРОВАНИЕ (ПРОГНОЗ НА 1 ДЕНЬ)")
print("="*80)

# Загрузка данных
df = pd.read_pickle(DATA_PATH / "clean_data.pkl")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")
print(f"Всего признаков: {len(df.columns)}")

print("\n⚠️ ВАЖНО: Исключены признаки с информационной утечкой:")
print("  - position_change (требует знания будущего)")
print("  - position_pct_change (требует знания будущего)")
print("  - motiv_change (требует знания будущего)")
print("  - motiv_pct_change (требует знания будущего)")
print("  - position_trend_7d (требует знания будущего)")
print("  - motiv_trend_7d (требует знания будущего)")

print("\nЦЕЛЕВАЯ ПЕРЕМЕННАЯ:")
print("  - position_delta_1d = изменение позиции через 1 день")
print("  - Отрицательное значение = улучшение (подъём в топ)")

# =====================================================
# 1. ПОДГОТОВКА ДАННЫХ (БЕЗ УТЕЧКИ)
# =====================================================
print("\n" + "="*60)
print("1. ПОДГОТОВКА ДАННЫХ ДЛЯ ML (БЕЗ УТЕЧКИ)")
print("="*60)

# Кодируем категориальные признаки
le_keyword = LabelEncoder()
df['keyword_encoded'] = le_keyword.fit_transform(df['keyword'])

# =====================================================
# РАСШИРЕННЫЙ СПИСОК ПРИЗНАКОВ
# =====================================================
# =====================================================
# ОБНОВЛЁННЫЙ СПИСОК ПРИЗНАКОВ (с фризами и обновлениями)
# =====================================================
feature_cols = [
    # Текущие значения
    'motiv', 'position', 'organic_us', 'activation_us',
    
    # Лаги
    'motiv_lag_1', 'position_lag_1',
    'motiv_lag_7', 'position_lag_7',
    'motiv_lag_14', 'position_lag_14',
    'motiv_lag_30', 'position_lag_30',
    
    # Скользящие средние
    'motiv_ma_3', 'position_ma_3',
    'motiv_ma_7', 'position_ma_7', 
    'motiv_std_7', 'position_std_7',
    'motiv_ma_14', 'position_ma_14',
    'motiv_ma_30', 'position_ma_30',
    
    # EWM
    'motiv_ewm_3', 'position_ewm_3',
    'motiv_ewm_7', 'position_ewm_7',
    'motiv_ewm_14', 'position_ewm_14',
    
    # Медианы
    'motiv_median_7', 'position_median_7',
    'motiv_median_14', 'position_median_14',
    
    # Min/Max
    'motiv_min_7', 'motiv_max_7',
    'position_min_7', 'position_max_7',
    'motiv_min_14', 'motiv_max_14',
    'position_min_14', 'position_max_14',
    
    # Ранги
    'position_rank', 'motiv_rank',
    
    # Изменения
    'motiv_change_1d', 'motiv_change_3d',
    'motiv_change_7d', 'motiv_change_14d',
    
    # Волатильность
    'position_volatility_7', 'position_volatility_14',
    
    # Отношения
    'motiv_position_ratio',
    'motiv_ma_diff', 'position_ma_diff',
    
    # Органика/активации
    'organic_ma_7', 'activation_ma_7',
    'organic_ma_14', 'activation_ma_14',
    'organic_std_7',
    
    # Временные
    'month', 'quarter', 'day_of_week', 'is_weekend',
    
    # 🆕 ФРИЗЫ И ОБНОВЛЕНИЯ
    'is_freeze',                    # Флаг фриза
    'days_to_freeze',               # Дней до ближайшего фриза
    'days_after_freeze',            # Дней после последнего фриза
    'is_ios_update_day',            # День обновления iOS
    'days_after_ios_update',        # Дней после обновления iOS
    'is_week_after_ios_update',     # Неделя после обновления
    'is_holiday_season',            # Праздничный сезон (дек-янв)
    
    # Категориальные
    'keyword_encoded'
]

# Фильтруем только те, которые есть в df
feature_cols = [col for col in feature_cols if col in df.columns]
print(f"  - Доступных признаков: {len(feature_cols)}")

# Целевая переменная - прогноз на 1 день
target_col = 'position_delta_1d'

print(f"  - Признаков (без утечки): {len(feature_cols)}")
print(f"  - Целевая переменная: {target_col}")

# Очистка данных
X = df[feature_cols].copy()
y = df[target_col].copy()

X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

mask = ~y.isna()
X = X[mask]
y = y[mask]

print(f"  - Данных для обучения: {len(X):,}")

# =====================================================
# 2. РАЗДЕЛЕНИЕ НА ВЫБОРКИ
# =====================================================
print("\n" + "="*60)
print("2. РАЗДЕЛЕНИЕ НА ОБУЧАЮЩУЮ И ТЕСТОВУЮ ВЫБОРКИ")
print("="*60)

df_ml_sorted = df.loc[mask].sort_values('date').reset_index(drop=True)
X_sorted = X.iloc[df_ml_sorted.index]
y_sorted = y.iloc[df_ml_sorted.index]

train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train = X_sorted[:train_size]
y_train = y_sorted[:train_size]
X_val = X_sorted[train_size:train_size + val_size]
y_val = y_sorted[train_size:train_size + val_size]
X_test = X_sorted[train_size + val_size:]
y_test = y_sorted[train_size + val_size:]

print(f"  - Train: {len(X_train):,} записей ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Val: {len(X_val):,} записей ({len(X_val)/len(X)*100:.1f}%)")
print(f"  - Test: {len(X_test):,} записей ({len(X_test)/len(X)*100:.1f}%)")

# =====================================================
# 3. СТАНДАРТИЗАЦИЯ
# =====================================================
print("\n" + "="*60)
print("3. СТАНДАРТИЗАЦИЯ ДАННЫХ")
print("="*60)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

print("  + Стандартизация выполнена")

# =====================================================
# 4. БАЗОВЫЕ МОДЕЛИ
# =====================================================
print("\n" + "="*60)
print("4. БАЗОВЫЕ МОДЕЛИ ДЛЯ СРАВНЕНИЯ")
print("="*60)

# 4.1. Linear Regression
print("\n4.1. Linear Regression")
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_scaled)
y_pred_lr = lr.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test_scaled, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_scaled, y_pred_lr))
r2_lr = r2_score(y_test_scaled, y_pred_lr)

print(f"  - MAE: {mae_lr:.4f}")
print(f"  - RMSE: {rmse_lr:.4f}")
print(f"  - R²: {r2_lr:.4f}")

# 4.2. Random Forest
print("\n4.2. Random Forest")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train_scaled)
y_pred_rf = rf.predict(X_test_scaled)

mae_rf = mean_absolute_error(y_test_scaled, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_scaled, y_pred_rf))
r2_rf = r2_score(y_test_scaled, y_pred_rf)

print(f"  - MAE: {mae_rf:.4f}")
print(f"  - RMSE: {rmse_rf:.4f}")
print(f"  - R²: {r2_rf:.4f}")

# =====================================================
# 5. XGBOOST С РЕГУЛЯРИЗАЦИЕЙ
# =====================================================
print("\n" + "="*60)
print("5. XGBOOST С УСИЛЕННОЙ РЕГУЛЯРИЗАЦИЕЙ")
print("="*60)

def objective_xgb_regularized(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 0.8),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.1, 10.0, log=True),
    }
    
    model = xgb.XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50,
        eval_metric='mae'
    )
    
    model.fit(
        X_train_scaled, y_train_scaled,
        eval_set=[(X_val_scaled, y_val_scaled)],
        verbose=False
    )
    
    y_pred = model.predict(X_val_scaled)
    return mean_absolute_error(y_val_scaled, y_pred)

print("\n  + Запуск оптимизации с регуляризацией (30 итераций)...")
study_xgb_reg = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb_reg.optimize(objective_xgb_regularized, n_trials=30, show_progress_bar=True)

print(f"\n  + Лучший MAE на валидации: {study_xgb_reg.best_value:.4f}")

# Обучаем финальную модель
xgb_reg = xgb.XGBRegressor(
    **study_xgb_reg.best_params,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='mae'
)

xgb_reg.fit(
    X_train_scaled, y_train_scaled,
    eval_set=[(X_val_scaled, y_val_scaled)],
    verbose=False
)

# Оценка на тесте
y_pred_xgb_reg = xgb_reg.predict(X_test_scaled)
mae_xgb_reg = mean_absolute_error(y_test_scaled, y_pred_xgb_reg)
rmse_xgb_reg = np.sqrt(mean_squared_error(y_test_scaled, y_pred_xgb_reg))
r2_xgb_reg = r2_score(y_test_scaled, y_pred_xgb_reg)

# Оценка на train
y_pred_xgb_reg_train = xgb_reg.predict(X_train_scaled)
mae_xgb_reg_train = mean_absolute_error(y_train_scaled, y_pred_xgb_reg_train)

print(f"\n  Результаты на тесте:")
print(f"    - MAE: {mae_xgb_reg:.4f}")
print(f"    - RMSE: {rmse_xgb_reg:.4f}")
print(f"    - R²: {r2_xgb_reg:.4f}")
print(f"\n  Проверка переобучения:")
print(f"    - Train MAE: {mae_xgb_reg_train:.4f}")
print(f"    - Test MAE: {mae_xgb_reg:.4f}")
print(f"    - Разница: {mae_xgb_reg_train - mae_xgb_reg:.4f}")

# =====================================================
# 6. LIGHTGBM С РЕГУЛЯРИЗАЦИЕЙ
# =====================================================
print("\n" + "="*60)
print("6. LIGHTGBM С РЕГУЛЯРИЗАЦИЕЙ")
print("="*60)

def objective_lgb_regularized(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.1, 1.0, log=True),
    }
    
    model = lgb.LGBMRegressor(**params, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        X_train_scaled, y_train_scaled,
        eval_set=[(X_val_scaled, y_val_scaled)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    y_pred = model.predict(X_val_scaled)
    return mean_absolute_error(y_val_scaled, y_pred)

print("\n  + Запуск оптимизации LightGBM (20 итераций)...")
study_lgb_reg = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb_reg.optimize(objective_lgb_regularized, n_trials=20, show_progress_bar=True)

print(f"\n  + Лучший MAE на валидации: {study_lgb_reg.best_value:.4f}")

lgb_reg = lgb.LGBMRegressor(**study_lgb_reg.best_params, random_state=42, n_jobs=-1, verbose=-1)
lgb_reg.fit(
    X_train_scaled, y_train_scaled,
    eval_set=[(X_val_scaled, y_val_scaled)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

y_pred_lgb_reg = lgb_reg.predict(X_test_scaled)
mae_lgb_reg = mean_absolute_error(y_test_scaled, y_pred_lgb_reg)
rmse_lgb_reg = np.sqrt(mean_squared_error(y_test_scaled, y_pred_lgb_reg))
r2_lgb_reg = r2_score(y_test_scaled, y_pred_lgb_reg)

print(f"\n  Результаты на тесте:")
print(f"    - MAE: {mae_lgb_reg:.4f}")
print(f"    - RMSE: {rmse_lgb_reg:.4f}")
print(f"    - R²: {r2_lgb_reg:.4f}")

# =====================================================
# 7. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("7. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*60)

results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'XGBoost (с регул.)',
        'LightGBM (с регул.)'
    ],
    'MAE': [mae_lr, mae_rf, mae_xgb_reg, mae_lgb_reg],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb_reg, rmse_lgb_reg],
    'R2': [r2_lr, r2_rf, r2_xgb_reg, r2_lgb_reg]
}).round(4)

print("\nРезультаты:")
print(results.to_string(index=False))

best_idx = results['MAE'].idxmin()
best_model_name = results.loc[best_idx, 'Model']
print(f"\n  + Лучшая модель: {best_model_name}")
print(f"    - MAE: {results.loc[best_idx, 'MAE']:.4f}")
print(f"    - RMSE: {results.loc[best_idx, 'RMSE']:.4f}")
print(f"    - R2: {results.loc[best_idx, 'R2']:.4f}")

# =====================================================
# 8. ВАЖНОСТЬ ПРИЗНАКОВ
# =====================================================
print("\n" + "="*60)
print("8. ВАЖНОСТЬ ПРИЗНАКОВ")
print("="*60)

# Группы признаков
feature_groups = {
    'Позиция': {
        'features': [f for f in feature_cols if 'position' in f],
        'color': '#FF6B6B'
    },
    'Мотив': {
        'features': [f for f in feature_cols if 'motiv' in f],
        'color': '#4ECDC4'
    },
    'Временные': {
        'features': ['month', 'quarter', 'day_of_week', 'is_weekend', 'day_of_year', 'week_of_year'],
        'color': '#45B7D1'
    },
    'Органика/Активации': {
        'features': [f for f in feature_cols if 'organic' in f or 'activation' in f],
        'color': '#96CEB4'
    },
    'Ключ': {
        'features': ['keyword_encoded'],
        'color': '#FFEAA7'
    }
}

def get_feature_group(feature_name):
    for group_name, group_info in feature_groups.items():
        if feature_name in group_info['features']:
            return group_name
    return 'Другие'

def get_feature_color(feature_name):
    for group_name, group_info in feature_groups.items():
        if feature_name in group_info['features']:
            return group_info['color']
    return '#D3D3D3'

importance = xgb_reg.feature_importances_
imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance,
    'importance_pct': importance * 100,
    'group': [get_feature_group(f) for f in feature_cols],
    'color': [get_feature_color(f) for f in feature_cols]
}).sort_values('importance', ascending=False)

print("\nТоп-20 важнейших признаков:")
print("  {:<30s} {:>12s} {:>12s} {:>20s}".format('Признак', 'Важность', 'Важность %', 'Группа'))
print("  " + "-"*80)

for i, row in imp_df.head(20).iterrows():
    print("  {:<30s} {:>12.4f} {:>11.2f}% {:>20s}".format(
        row['feature'][:30],
        row['importance'],
        row['importance_pct'],
        row['group']
    ))

# Группы
group_importance = imp_df.groupby('group')['importance'].sum().sort_values(ascending=False)
print("\nГруппы признаков:")
for group, imp in group_importance.items():
    pct = (imp / group_importance.sum()) * 100
    print(f"   - {group}: {pct:.1f}%")

# =====================================================
# 9. ВИЗУАЛИЗАЦИЯ
# =====================================================
print("\n" + "="*60)
print("9. ВИЗУАЛИЗАЦИЯ")
print("="*60)

plots_path = DATA_PATH / "plots"

# 9.1. Сравнение моделей
print("\n  + Построение графика сравнения моделей...")

names = results['Model'].tolist()
mae_vals = results['MAE'].tolist()
rmse_vals = results['RMSE'].tolist()
r2_vals = results['R2'].tolist()

colors = []
for name in names:
    if 'Linear' in name or 'Random' in name:
        colors.append('#B0B0B0')
    else:
        colors.append('#4ECDC4')

x = np.arange(len(names))
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Сравнение моделей: прогноз на 1 день', fontsize=16, fontweight='bold')

ax = axes[0]
bars = ax.bar(x, mae_vals, width, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Модель')
ax.set_ylabel('MAE')
ax.set_title('MAE (меньше = лучше)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

for i, (bar, val) in enumerate(zip(bars, mae_vals)):
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

best_idx = np.argmin(mae_vals)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

ax = axes[1]
bars = ax.bar(x, rmse_vals, width, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Модель')
ax.set_ylabel('RMSE')
ax.set_title('RMSE (меньше = лучше)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

for i, (bar, val) in enumerate(zip(bars, rmse_vals)):
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

best_idx = np.argmin(rmse_vals)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

ax = axes[2]
bars = ax.bar(x, r2_vals, width, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Модель')
ax.set_ylabel('R²')
ax.set_title('R² (больше = лучше)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

for i, (bar, val) in enumerate(zip(bars, r2_vals)):
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

best_idx = np.argmax(r2_vals)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig(plots_path / 'model_comparison_1day.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График сравнения моделей сохранён: {plots_path / 'model_comparison_1day.png'}")

# 9.2. Важность признаков
print("\n  + Построение графика важности признаков...")

fig, axes = plt.subplots(1, 2, figsize=(16, 12))
fig.suptitle('Важность признаков (прогноз на 1 день)', fontsize=16, fontweight='bold')

ax = axes[0]
imp_top25 = imp_df.head(25)
bar_colors = imp_top25['color'].tolist()

bars = ax.barh(imp_top25['feature'], imp_top25['importance'], 
               color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Важность')
ax.set_title('Топ-25 важнейших признаков')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

for i, (idx, row) in enumerate(imp_top25.iterrows()):
    ax.text(row['importance'] + 0.001, i, f'{row["importance"]:.3f}',
            va='center', fontsize=7)

ax = axes[1]
group_colors = {
    'Позиция': '#FF6B6B',
    'Мотив': '#4ECDC4',
    'Временные': '#45B7D1',
    'Органика/Активации': '#96CEB4',
    'Ключ': '#FFEAA7',
    'Другие': '#D3D3D3'
}

colors_group = [group_colors.get(g, '#D3D3D3') for g in group_importance.index]
bars = ax.barh(group_importance.index, group_importance.values, 
               color=colors_group, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Суммарная важность')
ax.set_title('Важность по группам признаков')
ax.grid(True, alpha=0.3, axis='x')

total_importance = group_importance.sum()
for i, (idx, val) in enumerate(group_importance.items()):
    pct = (val / total_importance) * 100
    ax.text(val + 0.005, i, f'{val:.4f} ({pct:.1f}%)',
            va='center', fontsize=10)

plt.tight_layout()
plt.savefig(plots_path / 'feature_importance_1day.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График важности признаков сохранён: {plots_path / 'feature_importance_1day.png'}")

# 9.3. Факт vs Прогноз
print("\n  + Построение графика факт vs прогноз...")

fig, ax = plt.subplots(figsize=(10, 8))

y_test_original = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1)).ravel()
y_pred_best = scaler_y.inverse_transform(y_pred_xgb_reg.reshape(-1, 1)).ravel()

ax.scatter(y_test_original, y_pred_best, alpha=0.4, s=15, color='steelblue')
ax.plot([y_test_original.min(), y_test_original.max()],
        [y_test_original.min(), y_test_original.max()],
        'r--', linewidth=2, label='Идеальное предсказание')
ax.set_xlabel('Фактическое изменение позиции (1 день)')
ax.set_ylabel('Предсказанное изменение позиции (1 день)')
ax.set_title(f'Факт vs Прогноз (XGBoost, прогноз на 1 день)')
ax.legend()
ax.grid(True, alpha=0.3)

ax.text(0.05, 0.95, f'MAE: {mae_xgb_reg:.4f}', transform=ax.transAxes,
        fontsize=12, verticalalignment='top')
ax.text(0.05, 0.90, f'R²: {r2_xgb_reg:.4f}', transform=ax.transAxes,
        fontsize=12, verticalalignment='top')

plt.tight_layout()
plt.savefig(plots_path / 'predictions_vs_actual_1day.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График факт vs прогноз сохранён: {plots_path / 'predictions_vs_actual_1day.png'}")

# =====================================================
# 10. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("10. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

results_path = DATA_PATH / "model_results_1day.csv"
results.to_csv(results_path, index=False)
print(f"  + Результаты сохранены: {results_path}")

model_path = DATA_PATH / "best_model_xgboost_1day.pkl"
joblib.dump(xgb_reg, model_path)
print(f"  + Лучшая модель сохранена: {model_path}")

scaler_path = DATA_PATH / "scaler_1day.pkl"
joblib.dump(scaler_X, scaler_path)
print(f"  + Scaler сохранён: {scaler_path}")

imp_df.to_csv(DATA_PATH / "feature_importance_1day.csv", index=False)
print(f"  + Важность признаков сохранена: {DATA_PATH / 'feature_importance_1day.csv'}")

joblib.dump(le_keyword, DATA_PATH / "le_keyword.pkl")
print(f"  + Кодировщик ключей сохранён")

# =====================================================
# 11. ИТОГОВЫЕ ВЫВОДЫ
# =====================================================
print("\n" + "="*60)
print("11. ИТОГОВЫЕ ВЫВОДЫ")
print("="*60)

print(f"""
📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ (ПРОГНОЗ НА 1 ДЕНЬ):

1. ЛУЧШАЯ МОДЕЛЬ: {best_model_name}
   - MAE:  {mae_xgb_reg:.4f}
   - RMSE: {rmse_xgb_reg:.4f}
   - R²:   {r2_xgb_reg:.4f}

2. ИНТЕРПРЕТАЦИЯ:
   - MAE = {mae_xgb_reg:.4f} → модель ошибается в среднем на {mae_xgb_reg:.2f} позиции
   - R² = {r2_xgb_reg:.4f} → модель объясняет {r2_xgb_reg*100:.1f}% вариации

3. ИСПОЛЬЗОВАНО ПРИЗНАКОВ: {len(feature_cols)}

4. ТОП-5 ВАЖНЕЙШИХ ПРИЗНАКОВ:
""")

for i, row in imp_df.head(5).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance_pct']:.2f}%")

print(f"""
5. ГРУППЫ ПРИЗНАКОВ:
""")
for group, imp in group_importance.items():
    pct = (imp / total_importance) * 100
    print(f"   - {group}: {pct:.1f}%")

print("""
6. РЕКОМЕНДАЦИИ:
   - Используйте XGBoost С РЕГУЛЯРИЗАЦИЕЙ для продакшена
   - Прогноз на 1 день даёт более точные краткосрочные предсказания
   - Основные драйверы: текущая позиция и её статистики
   - Мотив важен, но не является главным фактором
""")

print("\n" + "="*80)
print("✅ ШАГ 4 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 4: ML МОДЕЛИРОВАНИЕ (ПРОГНОЗ НА 1 ДЕНЬ)

Размер данных: 92,955 записей
Уникальных ключей: 134
Всего признаков: 121

⚠️ ВАЖНО: Исключены признаки с информационной утечкой:
  - position_change (требует знания будущего)
  - position_pct_change (требует знания будущего)
  - motiv_change (требует знания будущего)
  - motiv_pct_change (требует знания будущего)
  - position_trend_7d (требует знания будущего)
  - motiv_trend_7d (требует знания будущего)

ЦЕЛЕВАЯ ПЕРЕМЕННАЯ:
  - position_delta_1d = изменение позиции через 1 день
  - Отрицательное значение = улучшение (подъём в топ)

1. ПОДГОТОВКА ДАННЫХ ДЛЯ ML (БЕЗ УТЕЧКИ)
  - Доступных признаков: 68
  - Признаков (без утечки): 68
  - Целевая переменная: position_delta_1d
  - Данных для обучения: 92,955

2. РАЗДЕЛЕНИЕ НА ОБУЧАЮЩУЮ И ТЕСТОВУЮ ВЫБОРКИ
  - Train: 65,068 записей (70.0%)
  - Val: 13,943 записей (15.0%)
  - Test: 13,944 записей (15.0%)

3. СТАНДАРТИЗАЦИЯ ДАННЫХ
  + Стандартизация выполнена

4. БАЗОВЫЕ МОДЕЛИ ДЛЯ СРАВНЕНИЯ

4.1. 

[I 2026-09-12 11:03:42,514] A new study created in memory with name: no-name-24e03e0f-198a-4cfd-b722-0a604fecf9bf


  - MAE: 0.4917
  - RMSE: 0.9548
  - R²: 0.2283

5. XGBOOST С УСИЛЕННОЙ РЕГУЛЯРИЗАЦИЕЙ

  + Запуск оптимизации с регуляризацией (30 итераций)...


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-12 11:03:43,581] Trial 0 finished with value: 0.38859716310722375 and parameters: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.07259248719561363, 'subsample': 0.679597545259111, 'colsample_bytree': 0.5468055921327309, 'colsample_bylevel': 0.5467983561008608, 'min_child_weight': 5, 'reg_alpha': 13.39433470675048, 'reg_lambda': 6.054365855469246, 'gamma': 2.607024758370768}. Best is trial 0 with value: 0.38859716310722375.
[I 2026-09-12 11:03:44,035] Trial 1 finished with value: 0.4006820831163225 and parameters: {'n_estimators': 50, 'max_depth': 8, 'learning_rate': 0.09528587217040241, 'subsample': 0.5637017332034828, 'colsample_bytree': 0.5545474901621302, 'colsample_bylevel': 0.5550213529560302, 'min_child_weight': 9, 'reg_alpha': 4.816414530907083, 'reg_lambda': 3.6473162849112093, 'gamma': 0.38234752246751863}. Best is trial 0 with value: 0.38859716310722375.
[I 2026-09-12 11:03:44,459] Trial 2 finished with value: 0.40815148632563897 and parameters: {'n_estim

[I 2026-09-12 11:04:07,518] A new study created in memory with name: no-name-85fccc65-98fa-475f-936f-3495cbf3d2e2



  Результаты на тесте:
    - MAE: 0.4386
    - RMSE: 0.9105
    - R²: 0.2983

  Проверка переобучения:
    - Train MAE: 0.3171
    - Test MAE: 0.4386
    - Разница: -0.1216

6. LIGHTGBM С РЕГУЛЯРИЗАЦИЕЙ

  + Запуск оптимизации LightGBM (20 итераций)...


  0%|          | 0/20 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[150]	valid_0's l2: 0.769457
[I 2026-09-12 11:04:08,285] Trial 0 finished with value: 0.39376365891088244 and parameters: {'n_estimators': 150, 'num_leaves': 61, 'max_depth': 7, 'learning_rate': 0.050591436432963696, 'min_child_samples': 32, 'subsample': 0.5467983561008608, 'colsample_bytree': 0.5174250836504598, 'reg_alpha': 13.39433470675048, 'reg_lambda': 6.054365855469246, 'min_split_gain': 0.5105903209394755}. Best is trial 0 with value: 0.39376365891088244.
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[50]	valid_0's l2: 0.884376
[I 2026-09-12 11:04:08,670] Trial 1 finished with value: 0.40363501863957535 and parameters: {'n_estimators': 50, 'num_leaves': 62, 'max_depth': 7, 'learning_rate': 0.01777174904859463, 'min_child_samples': 34, 'subsample': 0.5550213529560302, 'colsample_bytree': 0.5912726728878613, 'r

# Шаг 5. Статистические тесты

In [6]:
# =====================================================
# ШАГ 5: СТАТИСТИЧЕСКИЕ ТЕСТЫ (ПРОГНОЗ НА 1 ДЕНЬ)
# =====================================================
print("\n" + "="*80)
print("ШАГ 5: СТАТИСТИЧЕСКИЕ ТЕСТЫ (ПРОГНОЗ НА 1 ДЕНЬ)")
print("="*80)

# Загрузка данных
df = pd.read_pickle(DATA_PATH / "clean_data.pkl")
motiv_efficiency_df = pd.read_csv(DATA_PATH / "motiv_efficiency.csv")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")
print(f"Всего признаков: {len(df.columns)}")

# Импорт статистических библиотек
from scipy import stats
from scipy.stats import (
    pearsonr, spearmanr, mannwhitneyu, kruskal, 
    shapiro, ttest_rel, ttest_ind, f_oneway
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML

print("\n✅ Все статистические библиотеки импортированы")

# =====================================================
# ФИЛЬТРАЦИЯ ДАННЫХ
# =====================================================
print("\n" + "="*60)
print("ФИЛЬТРАЦИЯ ДАННЫХ")
print("="*60)

start_date = pd.to_datetime('2026-06-20')
df_test = df[df['date'] >= start_date].copy()

print(f"\n  Исходный размер данных: {len(df):,} записей")
print(f"  Данных с 20.06.2026: {len(df_test):,} записей")
print(f"  Период: {df_test['date'].min()} - {df_test['date'].max()}")
print(f"  Уникальных ключей: {df_test['keyword'].nunique()}")

if len(df_test) == 0:
    print("\n  ⚠️ Нет данных после 20.06.2026! Используем последние 30% данных")
    df_test = df.sort_values('date').tail(int(len(df) * 0.3))
    print(f"  Взято последних 30% данных: {len(df_test):,} записей")

# Целевая переменная - прогноз на 1 день
TARGET_COL = 'position_delta_1d'
print(f"\n  🎯 Целевая метрика: {TARGET_COL} (прогноз на 1 день)")

# =====================================================
# СЛОВАРИ ДЛЯ РЕЗУЛЬТАТОВ
# =====================================================
scenario_names = {
    'stable': 'стабильный',
    'sharp_growth': 'резкий рост',
    'sharp_decline': 'резкий спад',
    'gentle_growth': 'плавный рост',
    'gentle_decline': 'плавный спад'
}

# =====================================================
# ГИПОТЕЗА 1: ПОХОЖИЕ ЗАПРОСЫ (КОРРЕЛЯЦИЯ > 0.7)
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 1: Похожие запросы (корреляция > 0.7)")
print("="*60)

def test_hypothesis_1(df_test):
    """
    H1: Корреляция позиций похожих ключей > 0.7
    """
    top_keywords = df_test['keyword'].value_counts().head(20).index.tolist()
    df_top = df_test[df_test['keyword'].isin(top_keywords)]
    
    pivot = df_top.pivot_table(
        index='date',
        columns='keyword',
        values='position'
    ).dropna(axis=1, how='all')
    
    if pivot.shape[1] < 2:
        return None
    
    pivot = pivot.fillna(pivot.mean())
    corr_matrix = pivot.corr()
    
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    corr_values = upper_tri.stack().values
    corr_values = corr_values[~np.isnan(corr_values)]
    
    if len(corr_values) == 0:
        return None
    
    t_stat, p_value = ttest_ind(corr_values, [0.7] * len(corr_values), alternative='less')
    
    corr_pairs = upper_tri.stack().sort_values(ascending=False).head(5)
    
    return {
        'n_keywords': len(top_keywords),
        'n_pairs': len(corr_values),
        'mean_corr': np.mean(corr_values),
        'median_corr': np.median(corr_values),
        'max_corr': np.max(corr_values),
        'min_corr': np.min(corr_values),
        't_stat': t_stat,
        'p_value': p_value,
        'top_pairs': corr_pairs.to_dict(),
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h1 = test_hypothesis_1(df_test)

if result_h1:
    print(f"\n📊 РЕЗУЛЬТАТЫ H1:")
    print(f"  - Ключей в анализе: {result_h1['n_keywords']}")
    print(f"  - Пар для корреляции: {result_h1['n_pairs']}")
    print(f"  - Средняя корреляция: {result_h1['mean_corr']:.3f}")
    print(f"  - Медианная корреляция: {result_h1['median_corr']:.3f}")
    print(f"  - Максимальная корреляция: {result_h1['max_corr']:.3f}")
    print(f"  - Минимальная корреляция: {result_h1['min_corr']:.3f}")
    print(f"  - t-статистика: {result_h1['t_stat']:.4f}")
    print(f"  - p-value: {result_h1['p_value']:.4f}")
    print(f"  - Результат: {result_h1['result']}")
    
    print(f"\n  Топ-5 пар с максимальной корреляцией:")
    for (k1, k2), corr in result_h1['top_pairs'].items():
        print(f"    - {k1[:25]} ↔ {k2[:25]}: {corr:.3f}")

# =====================================================
# ГИПОТЕЗА 2: СОГЛАСОВАННОСТЬ СЦЕНАРИЕВ
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 2: Согласованность сценариев мотива")
print("="*60)

def test_hypothesis_2(df_test):
    """H2: Сценарии изменения мотива согласованы между ключами"""
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    if len(df_analysis) < 50:
        return None
    
    scenario_effects = df_analysis.groupby('motiv_scenario')[TARGET_COL].agg(['mean', 'std', 'count'])
    scenario_effects = scenario_effects[scenario_effects['count'] >= 10]
    
    if len(scenario_effects) < 2:
        return None
    
    groups = [group[TARGET_COL].values for name, group in df_analysis.groupby('motiv_scenario') 
              if len(group) >= 10]
    
    if len(groups) < 2:
        return None
    
    h_stat, p_value = kruskal(*groups)
    
    return {
        'n_scenarios': len(scenario_effects),
        'scenario_effects': scenario_effects,
        'h_stat': h_stat,
        'p_value': p_value,
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h2 = test_hypothesis_2(df_test)

if result_h2:
    print(f"\n📊 РЕЗУЛЬТАТЫ H2:")
    print(f"  - Сценариев: {result_h2['n_scenarios']}")
    print(f"  - H-статистика: {result_h2['h_stat']:.4f}")
    print(f"  - p-value: {result_h2['p_value']:.4f}")
    print(f"  - Результат: {result_h2['result']}")
    
    print(f"\n  Эффективность сценариев (прогноз на 1 день):")
    print("  {:<20s} {:>15s} {:>15s} {:>10s}".format(
        'Сценарий', 'Среднее Δ', 'Стд.откл.', 'Кол-во'
    ))
    print("  " + "-"*65)
    for scenario, row in result_h2['scenario_effects'].iterrows():
        name = scenario_names.get(scenario, scenario)
        print("  {:<20s} {:>15.3f} {:>15.3f} {:>10.0f}".format(
            name, row['mean'], row['std'], row['count']
        ))

# =====================================================
# ГИПОТЕЗА 3: ЭФФЕКТИВНОСТЬ СЦЕНАРИЕВ
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 3: Эффективность сценариев продвижения")
print("="*60)

def test_hypothesis_3(df_test):
    """H3: Разные сценарии дают разный эффект на позицию"""
    df_scenarios = df_test[df_test['motiv_scenario'] != 'stable'].copy()
    df_scenarios = df_scenarios.dropna(subset=[TARGET_COL])
    
    if len(df_scenarios) < 20:
        return None
    
    groups = []
    scenario_stats = []
    
    for scenario, group in df_scenarios.groupby('motiv_scenario'):
        if len(group) >= 5:
            groups.append(group[TARGET_COL].values)
            scenario_stats.append({
                'scenario': scenario,
                'name': scenario_names.get(scenario, scenario),
                'mean': group[TARGET_COL].mean(),
                'median': group[TARGET_COL].median(),
                'std': group[TARGET_COL].std(),
                'count': len(group)
            })
    
    if len(groups) < 2:
        return None
    
    h_stat, p_value = kruskal(*groups)
    
    scenario_df = pd.DataFrame(scenario_stats)
    best_scenario = scenario_df.loc[scenario_df['mean'].idxmin()]
    worst_scenario = scenario_df.loc[scenario_df['mean'].idxmax()]
    
    return {
        'n_records': len(df_scenarios),
        'n_scenarios': len(groups),
        'scenario_stats': scenario_df,
        'h_stat': h_stat,
        'p_value': p_value,
        'best_scenario': best_scenario,
        'worst_scenario': worst_scenario,
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h3 = test_hypothesis_3(df_test)

if result_h3:
    print(f"\n📊 РЕЗУЛЬТАТЫ H3:")
    print(f"  - Записей: {result_h3['n_records']}")
    print(f"  - Сценариев: {result_h3['n_scenarios']}")
    print(f"  - H-статистика: {result_h3['h_stat']:.4f}")
    print(f"  - p-value: {result_h3['p_value']:.4f}")
    print(f"  - Результат: {result_h3['result']}")
    
    print(f"\n  📊 Лучший сценарий: {result_h3['best_scenario']['name']}")
    print(f"    - Среднее изменение: {result_h3['best_scenario']['mean']:.3f}")
    print(f"    - Количество: {result_h3['best_scenario']['count']:.0f}")
    
    print(f"\n  📊 Худший сценарий: {result_h3['worst_scenario']['name']}")
    print(f"    - Среднее изменение: {result_h3['worst_scenario']['mean']:.3f}")

# =====================================================
# ГИПОТЕЗА 4: ВЛИЯНИЕ ФРИЗОВ
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 4: Влияние фризов (мотив = 0)")
print("="*60)

def test_hypothesis_4(df_test):
    """H4: Отсутствие мотива (фриз) приводит к изменению позиции"""
    df_zero = df_test[df_test['motiv'] == 0].copy()
    df_nonzero = df_test[df_test['motiv'] > 0].copy()
    
    if len(df_zero) < 20 or len(df_nonzero) < 20:
        return None
    
    delta_zero = df_zero.dropna(subset=[TARGET_COL])[TARGET_COL]
    delta_nonzero = df_nonzero.dropna(subset=[TARGET_COL])[TARGET_COL]
    
    if len(delta_zero) < 10 or len(delta_nonzero) < 10:
        return None
    
    stat, p_value = mannwhitneyu(delta_zero, delta_nonzero, alternative='two-sided')
    
    n1, n2 = len(delta_zero), len(delta_nonzero)
    mean1, mean2 = delta_zero.mean(), delta_nonzero.mean()
    std1, std2 = delta_zero.std(), delta_nonzero.std()
    pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
    cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
    
    return {
        'n_zero': len(df_zero),
        'n_nonzero': len(df_nonzero),
        'mean_delta_zero': delta_zero.mean(),
        'mean_delta_nonzero': delta_nonzero.mean(),
        'median_delta_zero': delta_zero.median(),
        'median_delta_nonzero': delta_nonzero.median(),
        'stat': stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h4 = test_hypothesis_4(df_test)

if result_h4:
    print(f"\n📊 РЕЗУЛЬТАТЫ H4:")
    print(f"  - Записей с мотивом = 0: {result_h4['n_zero']:,}")
    print(f"  - Записей с мотивом > 0: {result_h4['n_nonzero']:,}")
    print(f"\n  Изменение позиции (прогноз на 1 день):")
    print(f"    - Без мотива:    среднее = {result_h4['mean_delta_zero']:.3f}, медиана = {result_h4['median_delta_zero']:.3f}")
    print(f"    - С мотивом > 0: среднее = {result_h4['mean_delta_nonzero']:.3f}, медиана = {result_h4['median_delta_nonzero']:.3f}")
    print(f"\n  - Mann-Whitney U статистика: {result_h4['stat']:.4f}")
    print(f"  - p-value: {result_h4['p_value']:.4f}")
    print(f"  - Cohen's d: {result_h4['cohens_d']:.3f}")
    print(f"  - Результат: {result_h4['result']}")
    
    if result_h4['mean_delta_nonzero'] < result_h4['mean_delta_zero']:
        print(f"\n  ✅ Мотив УЛУЧШАЕТ позицию (отрицательное изменение = улучшение)")
    else:
        print(f"\n  ⚠️ Мотив не даёт значимого эффекта")

# =====================================================
# ГИПОТЕЗА 5: ВЛИЯНИЕ МОТИВА НА ПОЗИЦИЮ
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 5: Влияние мотива на изменение позиции")
print("="*60)

def test_hypothesis_5(df_test):
    """H5: Мотив значимо влияет на изменение позиции"""
    df_analysis = df_test.dropna(subset=[TARGET_COL]).copy()
    
    if len(df_analysis) < 50:
        return None
    
    pearson_corr, p_pearson = pearsonr(df_analysis['motiv'], df_analysis[TARGET_COL])
    spearman_corr, p_spearman = spearmanr(df_analysis['motiv'], df_analysis[TARGET_COL])
    
    motiv_pos = df_analysis[df_analysis['motiv'] > 0][TARGET_COL]
    motiv_zero = df_analysis[df_analysis['motiv'] == 0][TARGET_COL]
    
    cohens_d = 0
    p_mw = 1.0
    
    if len(motiv_pos) > 0 and len(motiv_zero) > 0:
        stat, p_mw = mannwhitneyu(motiv_pos, motiv_zero)
        n1, n2 = len(motiv_pos), len(motiv_zero)
        mean1, mean2 = motiv_pos.mean(), motiv_zero.mean()
        std1, std2 = motiv_pos.std(), motiv_zero.std()
        pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
        cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
    
    hypothesis_accepted = p_pearson < 0.05 or p_spearman < 0.05
    
    return {
        'n_records': len(df_analysis),
        'pearson_corr': pearson_corr,
        'p_pearson': p_pearson,
        'spearman_corr': spearman_corr,
        'p_spearman': p_spearman,
        'p_mw': p_mw,
        'cohens_d': cohens_d,
        'mean_motiv_pos': motiv_pos.mean() if len(motiv_pos) > 0 else 0,
        'mean_motiv_zero': motiv_zero.mean() if len(motiv_zero) > 0 else 0,
        'hypothesis_accepted': hypothesis_accepted,
        'result': '✅ ПОДТВЕРЖДЕНА' if hypothesis_accepted else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h5 = test_hypothesis_5(df_test)

if result_h5:
    print(f"\n📊 РЕЗУЛЬТАТЫ H5:")
    print(f"  - Записей: {result_h5['n_records']:,}")
    print(f"\n  Корреляции (мотив vs изменение позиции):")
    print(f"    - Pearson:  r = {result_h5['pearson_corr']:.4f}, p = {result_h5['p_pearson']:.4f}")
    print(f"    - Spearman: r = {result_h5['spearman_corr']:.4f}, p = {result_h5['p_spearman']:.4f}")
    print(f"\n  Сравнение групп:")
    print(f"    - С мотивом:    среднее Δ = {result_h5['mean_motiv_pos']:.3f}")
    print(f"    - Без мотива:   среднее Δ = {result_h5['mean_motiv_zero']:.3f}")
    print(f"    - Mann-Whitney p-value: {result_h5['p_mw']:.4f}")
    print(f"    - Cohen's d: {result_h5['cohens_d']:.3f}")
    print(f"\n  - Результат: {result_h5['result']}")

# =====================================================
# ГИПОТЕЗА 6: ЭФФЕКТИВНОСТЬ МОТИВА (НОВАЯ)
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 6: Эффективность мотива > 0")
print("="*60)

def test_hypothesis_6(df_test, motiv_efficiency_df):
    """H6: Средняя эффективность мотива значимо > 0"""
    if motiv_efficiency_df is None or len(motiv_efficiency_df) == 0:
        return None
    
    eff_values = motiv_efficiency_df['efficiency'].dropna()
    eff_values = eff_values[(eff_values > -10) & (eff_values < 50)]
    
    if len(eff_values) < 10:
        return None
    
    t_stat, p_value = ttest_ind(eff_values, [0] * len(eff_values), alternative='greater')
    
    positive_count = (eff_values > 0).sum()
    
    return {
        'n_keywords': len(eff_values),
        'mean_efficiency': eff_values.mean(),
        'median_efficiency': eff_values.median(),
        'std_efficiency': eff_values.std(),
        'max_efficiency': eff_values.max(),
        'min_efficiency': eff_values.min(),
        'positive_count': positive_count,
        'positive_pct': positive_count / len(eff_values) * 100,
        't_stat': t_stat,
        'p_value': p_value,
        'hypothesis_accepted': p_value < 0.05 and eff_values.mean() > 0,
        'result': '✅ ПОДТВЕРЖДЕНА' if (p_value < 0.05 and eff_values.mean() > 0) else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h6 = test_hypothesis_6(df_test, motiv_efficiency_df)

if result_h6:
    print(f"\n📊 РЕЗУЛЬТАТЫ H6:")
    print(f"  - Ключей проанализировано: {result_h6['n_keywords']}")
    print(f"  - Средняя эффективность: {result_h6['mean_efficiency']:.3f}")
    print(f"  - Медианная эффективность: {result_h6['median_efficiency']:.3f}")
    print(f"  - Стандартное отклонение: {result_h6['std_efficiency']:.3f}")
    print(f"  - Максимум: {result_h6['max_efficiency']:.3f}")
    print(f"  - Минимум: {result_h6['min_efficiency']:.3f}")
    print(f"\n  - Ключей с положительной эффективностью: {result_h6['positive_count']} ({result_h6['positive_pct']:.1f}%)")
    print(f"  - t-статистика: {result_h6['t_stat']:.4f}")
    print(f"  - p-value: {result_h6['p_value']:.4f}")
    print(f"  - Результат: {result_h6['result']}")

# =====================================================
# ГИПОТЕЗА 7: ВЛИЯНИЕ ФРИЗОВ НА ПРОДВИЖЕНИЕ (НОВАЯ)
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 7: Влияние фризов на продвижение")
print("="*60)

def test_hypothesis_7(df_test):
    """H7: В периоды фризов мотив не работает"""
    if 'is_freeze' not in df_test.columns:
        return None
    
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    in_freeze = df_analysis[df_analysis['is_freeze'] == 1][TARGET_COL]
    out_freeze = df_analysis[df_analysis['is_freeze'] == 0][TARGET_COL]
    
    if len(in_freeze) < 10 or len(out_freeze) < 10:
        return None
    
    stat, p_value = mannwhitneyu(in_freeze, out_freeze, alternative='two-sided')
    
    n1, n2 = len(in_freeze), len(out_freeze)
    mean1, mean2 = in_freeze.mean(), out_freeze.mean()
    std1, std2 = in_freeze.std(), out_freeze.std()
    pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
    cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
    
    return {
        'n_in_freeze': len(in_freeze),
        'n_out_freeze': len(out_freeze),
        'mean_in_freeze': mean1,
        'mean_out_freeze': mean2,
        'diff': mean1 - mean2,
        'stat': stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h7 = test_hypothesis_7(df_test)

if result_h7:
    print(f"\n📊 РЕЗУЛЬТАТЫ H7:")
    print(f"  - Записей во фризе: {result_h7['n_in_freeze']:,}")
    print(f"  - Записей вне фриза: {result_h7['n_out_freeze']:,}")
    print(f"\n  Изменение позиции (прогноз на 1 день):")
    print(f"    - Во фризе:    среднее = {result_h7['mean_in_freeze']:.3f}")
    print(f"    - Вне фриза:   среднее = {result_h7['mean_out_freeze']:.3f}")
    print(f"    - Разница:     {result_h7['diff']:.3f}")
    print(f"\n  - Mann-Whitney U: {result_h7['stat']:.4f}")
    print(f"  - p-value: {result_h7['p_value']:.4f}")
    print(f"  - Cohen's d: {result_h7['cohens_d']:.3f}")
    print(f"  - Результат: {result_h7['result']}")
    
    if result_h7['mean_in_freeze'] > result_h7['mean_out_freeze']:
        print(f"\n  ⚠️ Во время фризов позиции УХУДШАЮТСЯ (ожидаемо)")
    else:
        print(f"\n  ✅ Во время фризов позиции стабильны")

# =====================================================
# ГИПОТЕЗА 8: ВЛИЯНИЕ iOS ОБНОВЛЕНИЙ (НОВАЯ)
# =====================================================
print("\n" + "="*60)
print("ГИПОТЕЗА 8: Влияние iOS обновлений")
print("="*60)

def test_hypothesis_8(df_test):
    """H8: Неделя после iOS обновления даёт изменения в позициях"""
    if 'is_week_after_ios_update' not in df_test.columns:
        return None
    
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    week_after = df_analysis[df_analysis['is_week_after_ios_update'] == 1][TARGET_COL]
    normal_days = df_analysis[df_analysis['is_week_after_ios_update'] == 0][TARGET_COL]
    
    if len(week_after) < 10 or len(normal_days) < 10:
        return None
    
    stat, p_value = mannwhitneyu(week_after, normal_days, alternative='two-sided')
    
    n1, n2 = len(week_after), len(normal_days)
    mean1, mean2 = week_after.mean(), normal_days.mean()
    std1, std2 = week_after.std(), normal_days.std()
    pooled_std = np.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
    cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0
    
    return {
        'n_week_after': len(week_after),
        'n_normal': len(normal_days),
        'mean_week_after': mean1,
        'mean_normal': mean2,
        'diff': mean1 - mean2,
        'stat': stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'hypothesis_accepted': p_value < 0.05,
        'result': '✅ ПОДТВЕРЖДЕНА' if p_value < 0.05 else '❌ НЕ ПОДТВЕРЖДЕНА'
    }

result_h8 = test_hypothesis_8(df_test)

if result_h8:
    print(f"\n📊 РЕЗУЛЬТАТЫ H8:")
    print(f"  - Записей за неделю после обновления: {result_h8['n_week_after']:,}")
    print(f"  - Записей в обычные дни: {result_h8['n_normal']:,}")
    print(f"\n  Изменение позиции (прогноз на 1 день):")
    print(f"    - Неделя после обновления: {result_h8['mean_week_after']:.3f}")
    print(f"    - Обычные дни:             {result_h8['mean_normal']:.3f}")
    print(f"    - Разница:                 {result_h8['diff']:.3f}")
    print(f"\n  - Mann-Whitney U: {result_h8['stat']:.4f}")
    print(f"  - p-value: {result_h8['p_value']:.4f}")
    print(f"  - Cohen's d: {result_h8['cohens_d']:.3f}")
    print(f"  - Результат: {result_h8['result']}")

# =====================================================
# СВОДНАЯ ТАБЛИЦА ГИПОТЕЗ
# =====================================================
print("\n" + "="*60)
print("СВОДНАЯ ТАБЛИЦА ГИПОТЕЗ")
print("="*60)

hypotheses_summary = []

if result_h1:
    hypotheses_summary.append({
        'Гипотеза': 'H1: Похожие запросы',
        'Метрика': f"Средняя корреляция: {result_h1['mean_corr']:.3f}",
        'p-value': f"{result_h1['p_value']:.4f}",
        'Результат': result_h1['result']
    })

if result_h2:
    hypotheses_summary.append({
        'Гипотеза': 'H2: Согласованность сценариев',
        'Метрика': f"Сценариев: {result_h2['n_scenarios']}",
        'p-value': f"{result_h2['p_value']:.4f}",
        'Результат': result_h2['result']
    })

if result_h3:
    hypotheses_summary.append({
        'Гипотеза': 'H3: Эффективность сценариев',
        'Метрика': f"Лучший: {result_h3['best_scenario']['name']}",
        'p-value': f"{result_h3['p_value']:.4f}",
        'Результат': result_h3['result']
    })

if result_h4:
    hypotheses_summary.append({
        'Гипотеза': 'H4: Влияние фризов (мотив=0)',
        'Метрика': f"Cohen's d: {result_h4['cohens_d']:.3f}",
        'p-value': f"{result_h4['p_value']:.4f}",
        'Результат': result_h4['result']
    })

if result_h5:
    hypotheses_summary.append({
        'Гипотеза': 'H5: Влияние мотива',
        'Метрика': f"Spearman: {result_h5['spearman_corr']:.3f}",
        'p-value': f"{result_h5['p_spearman']:.4f}",
        'Результат': result_h5['result']
    })

if result_h6:
    hypotheses_summary.append({
        'Гипотеза': 'H6: Эффективность мотива',
        'Метрика': f"Средняя: {result_h6['mean_efficiency']:.3f}",
        'p-value': f"{result_h6['p_value']:.4f}",
        'Результат': result_h6['result']
    })

if result_h7:
    hypotheses_summary.append({
        'Гипотеза': 'H7: Влияние фризов (события)',
        'Метрика': f"Δ во фризе: {result_h7['mean_in_freeze']:.3f}",
        'p-value': f"{result_h7['p_value']:.4f}",
        'Результат': result_h7['result']
    })

if result_h8:
    hypotheses_summary.append({
        'Гипотеза': 'H8: Влияние iOS обновлений',
        'Метрика': f"Δ после обновления: {result_h8['mean_week_after']:.3f}",
        'p-value': f"{result_h8['p_value']:.4f}",
        'Результат': result_h8['result']
    })

summary_df = pd.DataFrame(hypotheses_summary)

if len(summary_df) > 0:
    def highlight_result(val):
        if '✅' in str(val):
            return 'background-color: #90EE90; color: #006400; font-weight: bold'
        elif '❌' in str(val):
            return 'background-color: #FFB6C1; color: #8B0000; font-weight: bold'
        return ''
    
    styled_df = summary_df.style.map(highlight_result, subset=['Результат'])
    styled_df = styled_df.set_caption("📊 Результаты проверки гипотез (прогноз на 1 день)")
    display(styled_df)
    
    summary_df.to_csv(DATA_PATH / "hypotheses_summary_1day.csv", index=False)
    print(f"\n  + Сводная таблица сохранена: {DATA_PATH / 'hypotheses_summary_1day.csv'}")

# =====================================================
# ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*60)

plots_path = DATA_PATH / "plots"

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('Результаты статистических тестов (прогноз на 1 день)', fontsize=16, fontweight='bold')

# 1. H1: Корреляции
ax = axes[0, 0]
if result_h1:
    ax.hist(result_h1['top_pairs'].values(), bins=10, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(x=0.7, color='red', linestyle='--', linewidth=2, label='Порог 0.7')
    ax.set_xlabel('Корреляция')
    ax.set_ylabel('Количество пар')
    ax.set_title('H1: Топ-5 пар корреляций')
    ax.legend()
    ax.grid(True, alpha=0.3)

# 2. H2: Эффективность сценариев
ax = axes[0, 1]
if result_h2:
    scenario_eff = result_h2['scenario_effects']
    names = [scenario_names.get(s, s) for s in scenario_eff.index]
    colors = ['green' if m < 0 else 'red' for m in scenario_eff['mean']]
    bars = ax.barh(names, scenario_eff['mean'], color=colors, alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Среднее изменение позиции')
    ax.set_title('H2: Эффективность сценариев')
    ax.grid(True, alpha=0.3, axis='x')

# 3. H3: Лучший сценарий
ax = axes[0, 2]
if result_h3:
    scenario_df = result_h3['scenario_stats']
    colors = ['gold' if row['scenario'] == result_h3['best_scenario']['scenario'] else 'steelblue' 
              for _, row in scenario_df.iterrows()]
    bars = ax.barh(scenario_df['name'], scenario_df['mean'], color=colors, alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Среднее изменение позиции')
    ax.set_title('H3: Сравнение сценариев')
    ax.grid(True, alpha=0.3, axis='x')

# 4. H4: Влияние фризов (мотив=0)
ax = axes[0, 3]
if result_h4:
    categories = ['Без мотива', 'С мотивом']
    values = [result_h4['mean_delta_zero'], result_h4['mean_delta_nonzero']]
    colors = ['red' if v > 0 else 'green' for v in values]
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Среднее изменение позиции')
    ax.set_title(f'H4: Влияние фризов (p={result_h4["p_value"]:.4f})')
    ax.grid(True, alpha=0.3, axis='y')

# 5. H5: Корреляция мотива
ax = axes[1, 0]
if result_h5:
    categories = ['Pearson', 'Spearman']
    values = [result_h5['pearson_corr'], result_h5['spearman_corr']]
    colors = ['green' if v < 0 else 'red' for v in values]
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Корреляция')
    ax.set_title('H5: Корреляция мотива и Δ позиции')
    ax.grid(True, alpha=0.3, axis='y')

# 6. H6: Эффективность мотива
ax = axes[1, 1]
if result_h6:
    ax.hist(motiv_efficiency_df['efficiency'].dropna(), bins=30, 
            color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Нулевая эффективность')
    ax.axvline(x=result_h6['mean_efficiency'], color='green', linestyle='--', 
               linewidth=2, label=f'Средняя: {result_h6["mean_efficiency"]:.3f}')
    ax.set_xlabel('Эффективность')
    ax.set_ylabel('Количество ключей')
    ax.set_title('H6: Распределение эффективности мотива')
    ax.legend()
    ax.grid(True, alpha=0.3)

# 7. H7: Влияние фризов (НОВАЯ)
ax = axes[1, 2]
if result_h7:
    categories = ['Во фризе', 'Вне фриза']
    values = [result_h7['mean_in_freeze'], result_h7['mean_out_freeze']]
    colors = ['purple', 'steelblue']
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Среднее изменение позиции')
    ax.set_title(f'H7: Влияние фризов (p={result_h7["p_value"]:.4f})')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., val + (0.02 if val >= 0 else -0.02),
                f'{val:.3f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

# 8. H8: Влияние iOS обновлений (НОВАЯ)
ax = axes[1, 3]
if result_h8:
    categories = ['Неделя после\nобновления', 'Обычные дни']
    values = [result_h8['mean_week_after'], result_h8['mean_normal']]
    colors = ['orange', 'steelblue']
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Среднее изменение позиции')
    ax.set_title(f'H8: Влияние iOS (p={result_h8["p_value"]:.4f})')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., val + (0.02 if val >= 0 else -0.02),
                f'{val:.3f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.savefig(plots_path / 'hypotheses_1day.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График гипотез сохранён: {plots_path / 'hypotheses_1day.png'}")

# =====================================================
# ИТОГОВЫЕ ВЫВОДЫ
# =====================================================
print("\n" + "="*60)
print("ИТОГОВЫЕ ВЫВОДЫ ПО ГИПОТЕЗАМ")
print("="*60)

if len(summary_df) > 0:
    accepted = (summary_df['Результат'].str.contains('✅')).sum()
    total = len(summary_df)
    pct = accepted / total * 100 if total > 0 else 0
    
    print(f"\n📊 ОБОБЩЁННЫЕ РЕЗУЛЬТАТЫ:")
    print(f"  - Всего гипотез проверено: {total}")
    print(f"  - Подтверждено: {accepted}")
    print(f"  - Процент подтверждения: {pct:.1f}%")
    
    print(f"\n📋 ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
    for _, row in summary_df.iterrows():
        print(f"\n  {row['Гипотеза']}")
        print(f"    Метрика: {row['Метрика']}")
        print(f"    p-value: {row['p-value']}")
        print(f"    Результат: {row['Результат']}")

print("\n" + "="*80)
print("✅ ШАГ 5 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 5: СТАТИСТИЧЕСКИЕ ТЕСТЫ (ПРОГНОЗ НА 1 ДЕНЬ)

Размер данных: 92,955 записей
Уникальных ключей: 134
Всего признаков: 121

✅ Все статистические библиотеки импортированы

ФИЛЬТРАЦИЯ ДАННЫХ

  Исходный размер данных: 92,955 записей
  Данных с 20.06.2026: 2,978 записей
  Период: 2026-06-20 00:00:00 - 2026-07-15 00:00:00
  Уникальных ключей: 117

  🎯 Целевая метрика: position_delta_1d (прогноз на 1 день)

ГИПОТЕЗА 1: Похожие запросы (корреляция > 0.7)

📊 РЕЗУЛЬТАТЫ H1:
  - Ключей в анализе: 20
  - Пар для корреляции: 190
  - Средняя корреляция: 0.980
  - Медианная корреляция: 0.988
  - Максимальная корреляция: 1.000
  - Минимальная корреляция: 0.919
  - t-статистика: 196.7359
  - p-value: 1.0000
  - Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  Топ-5 пар с максимальной корреляцией:
    - audio amplifier ↔ audio enhancer: 1.000
    - amplify ↔ audio amplifier: 1.000
    - amplify ↔ audio enhancer: 1.000
    - bass boost ↔ bass booster free: 0.999
    - audio amplifier ↔ bass boost: 0.998

ГИПОТЕЗА 2: С

,Гипотеза,Метрика,p-value,Результат
0,H1: Похожие запросы,Средняя корреляция: 0.980,1.0000,❌ НЕ ПОДТВЕРЖДЕНА
1,H2: Согласованность сценариев,Сценариев: 5,0.3803,❌ НЕ ПОДТВЕРЖДЕНА
2,H3: Эффективность сценариев,Лучший: резкий рост,0.5390,❌ НЕ ПОДТВЕРЖДЕНА
3,H4: Влияние фризов (мотив=0),Cohen's d: -0.083,0.2584,❌ НЕ ПОДТВЕРЖДЕНА
4,H5: Влияние мотива,Spearman: -0.022,0.2408,❌ НЕ ПОДТВЕРЖДЕНА
5,H6: Эффективность мотива,Средняя: 8.887,0.0000,✅ ПОДТВЕРЖДЕНА
6,H7: Влияние фризов (события),Δ во фризе: 0.000,0.0000,✅ ПОДТВЕРЖДЕНА
7,H8: Влияние iOS обновлений,Δ после обновления: 0.000,0.0033,✅ ПОДТВЕРЖДЕНА



  + Сводная таблица сохранена: D:\denis\APP\app_new\data\hypotheses_summary_1day.csv

ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
  + График гипотез сохранён: D:\denis\APP\app_new\data\plots\hypotheses_1day.png

ИТОГОВЫЕ ВЫВОДЫ ПО ГИПОТЕЗАМ

📊 ОБОБЩЁННЫЕ РЕЗУЛЬТАТЫ:
  - Всего гипотез проверено: 8
  - Подтверждено: 3
  - Процент подтверждения: 37.5%

📋 ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ:

  H1: Похожие запросы
    Метрика: Средняя корреляция: 0.980
    p-value: 1.0000
    Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  H2: Согласованность сценариев
    Метрика: Сценариев: 5
    p-value: 0.3803
    Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  H3: Эффективность сценариев
    Метрика: Лучший: резкий рост
    p-value: 0.5390
    Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  H4: Влияние фризов (мотив=0)
    Метрика: Cohen's d: -0.083
    p-value: 0.2584
    Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  H5: Влияние мотива
    Метрика: Spearman: -0.022
    p-value: 0.2408
    Результат: ❌ НЕ ПОДТВЕРЖДЕНА

  H6: Эффективность мотива
    Метрика: Средняя: 8.887
    p-value: 0.0000
 

# Шаг6. Провека правил

In [7]:
# =====================================================
# ШАГ 6: ПРОВЕРКА ПРАВИЛ ИЗ ДОКУМЕНТА
# =====================================================
print("\n" + "="*80)
print("ШАГ 6: ПРОВЕРКА ПРАВИЛ ИЗ ДОКУМЕНТА (ПРОГНОЗ НА 1 ДЕНЬ)")
print("="*80)

# Загрузка данных
df = pd.read_pickle(DATA_PATH / "clean_data.pkl")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")
print(f"Всего признаков: {len(df.columns)}")

from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Целевая переменная - прогноз на 1 день
TARGET_COL = 'position_delta_1d'

# Фильтрация данных
start_date = pd.to_datetime('2026-06-20')
df_test = df[df['date'] >= start_date].copy()

if len(df_test) == 0:
    print("\n  ⚠️ Нет данных после 20.06.2026! Используем последние 30% данных")
    df_test = df.sort_values('date').tail(int(len(df) * 0.3))

print(f"\nАнализируемый период: {df_test['date'].min()} - {df_test['date'].max()}")
print(f"Записей: {len(df_test):,}")

# =====================================================
# ОПРЕДЕЛЕНИЕ ПРАВИЛ
# =====================================================
print("\n" + "="*60)
print("1. ОПРЕДЕЛЕНИЕ ПРАВИЛ ИЗ ДОКУМЕНТА")
print("="*60)

rules = {
    'R1': {
        'name': 'Базовый цикл продвижения',
        'description': 'M = 1 → 1 → 0 (два дня мотива, день пропуска)',
        'check': 'Проверка: после двух дней мотива 1 и пропуска позиция улучшается'
    },
    'R2': {
        'name': 'Оценка динамики после 2-3 дней',
        'description': 'Рост = улучшение на 1+ позиций, Падение = ухудшение на 2+ позиций',
        'check': 'Проверка: мотив > 0 даёт отрицательное изменение позиции'
    },
    'R3': {
        'name': 'Увеличение мотива при росте > 10 позиций',
        'description': 'Если позиция выросла > 10, увеличиваем мотив до 2',
        'check': 'Проверка: при сильном росте мотив увеличивается'
    },
    'R4': {
        'name': 'Пропуск при падении',
        'description': 'При падении позиции делаем пропуск минимум на 1 день',
        'check': 'Проверка: при падении мотив снижается до 0'
    },
    'R5': {
        'name': 'Ограничение мотива до 2 для новых запросов',
        'description': 'Для новых запросов мотив не более 2',
        'check': 'Проверка: мотив не превышает 2'
    },
    'R6': {
        'name': 'Целевые позиции по частотности',
        'description': 'ВЧ/СЧ: до 40 позиции, НЧ: до 20 позиции',
        'check': 'Проверка: позиция стремится к целевой'
    },
    'R7': {
        'name': 'Максимум 3 цикла с мотивом 2',
        'description': 'После 3 циклов 2→2→0 без роста > 50% - остановка',
        'check': 'Проверка: эффективность циклов с мотивом 2'
    },
    'R8': {
        'name': 'Повышение мотива до 3 при росте 50%',
        'description': 'Если позиция выросла на 50% к цели, повышаем до 3',
        'check': 'Проверка: мотив 3 даёт улучшение'
    },
    'R9': {
        'name': 'Процентный шаг (15%/10%)',
        'description': 'Увеличение мотива на 15% (1-14) или 10% (15+)',
        'check': 'Проверка: мотив растёт по правилу'
    },
    'R10': {
        'name': 'Откат -30% при отсутствии роста',
        'description': 'При отсутствии роста снижаем мотив на 30%',
        'check': 'Проверка: откат даёт восстановление'
    },
    'R11': {
        'name': 'Поддержка снижением на 2-3 единицы',
        'description': 'После достижения цели снижаем мотив на 2-3',
        'check': 'Проверка: поддержка удерживает позицию'
    },
    'R12': {
        'name': 'Максимальный мотив 30% от трафика',
        'description': 'Мотив не должен превышать 30% от органического трафика',
        'check': 'Проверка: мотив не превышает 30%'
    },
    'R13': {
        'name': '🆕 Пауза во время фризов',
        'description': 'Во время фризов продвижение не работает - нужно ставить паузу',
        'check': 'Проверка: во время фризов мотив не работает'
    },
    'R14': {
        'name': '🆕 Усиление после iOS обновлений',
        'description': 'В неделю после iOS обновления - пик активности пользователей',
        'check': 'Проверка: неделя после обновления даёт лучший результат'
    }
}

print("\nОпределены правила из документа:")
for rule_id, rule in rules.items():
    print(f"  {rule_id}: {rule['name']}")

# =====================================================
# ФУНКЦИИ ПРОВЕРКИ ПРАВИЛ
# =====================================================

def check_rule_r1(df_test):
    """R1: Базовый цикл 1→1→0"""
    df_sorted = df_test.sort_values(['keyword', 'date'])
    results = []
    
    for keyword in df_test['keyword'].unique()[:20]:
        kw_data = df_sorted[df_sorted['keyword'] == keyword]
        if len(kw_data) < 6:
            continue
        
        motiv_pattern = kw_data['motiv'].values[:10]
        for i in range(len(motiv_pattern)-3):
            if motiv_pattern[i] == 1 and motiv_pattern[i+1] == 1 and motiv_pattern[i+2] == 0:
                if i+3 < len(kw_data):
                    delta = kw_data['position'].values[i+3] - kw_data['position'].values[i]
                    results.append({
                        'keyword': keyword,
                        'position_delta': delta,
                        'improved': delta < 0
                    })
                break
    
    if results:
        improved = sum(1 for r in results if r['improved'])
        return {
            'n_patterns': len(results),
            'improved': improved,
            'pct_improved': improved/len(results)*100 if results else 0,
            'result': '✅ ПОДТВЕРЖДЕНО' if improved/len(results) > 0.5 else '❌ НЕ ПОДТВЕРЖДЕНО'
        }
    return None

def check_rule_r2(df_test):
    """R2: Мотив > 0 даёт улучшение позиции"""
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    if len(df_analysis) < 20:
        return None
    
    motiv_positive = df_analysis[df_analysis['motiv'] > 0][TARGET_COL]
    motiv_zero = df_analysis[df_analysis['motiv'] == 0][TARGET_COL]
    
    if len(motiv_positive) < 5 or len(motiv_zero) < 5:
        return None
    
    stat, p_value = mannwhitneyu(motiv_positive, motiv_zero)
    
    mean_positive = motiv_positive.mean()
    mean_zero = motiv_zero.mean()
    effect = mean_positive < mean_zero
    
    return {
        'n_positive': len(motiv_positive),
        'n_zero': len(motiv_zero),
        'mean_positive': mean_positive,
        'mean_zero': mean_zero,
        'diff': mean_positive - mean_zero,
        'p_value': p_value,
        'significant': p_value < 0.05,
        'effect': effect,
        'result': '✅ ПОДТВЕРЖДЕНО' if (effect and p_value < 0.05) else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r3(df_test):
    """R3: Сильный рост (>10) → увеличение мотива"""
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    if len(df_analysis) < 20:
        return None
    
    strong_growth = df_analysis[df_analysis[TARGET_COL] < -10]
    
    if len(strong_growth) < 3:
        return None
    
    avg_motiv_after = strong_growth['motiv'].mean()
    avg_motiv_before = df_analysis['motiv'].mean()
    
    return {
        'n_strong_growth': len(strong_growth),
        'avg_motiv_after': avg_motiv_after,
        'avg_motiv_before': avg_motiv_before,
        'motiv_increased': avg_motiv_after > avg_motiv_before,
        'result': '✅ ПОДТВЕРЖДЕНО' if avg_motiv_after > avg_motiv_before else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r4(df_test):
    """R4: При падении → мотив = 0"""
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    if len(df_analysis) < 20:
        return None
    
    falling = df_analysis[df_analysis[TARGET_COL] > 1]
    
    if len(falling) < 3:
        return None
    
    avg_motiv_falling = falling['motiv'].mean()
    avg_motiv_other = df_analysis[df_analysis[TARGET_COL] <= 1]['motiv'].mean()
    
    return {
        'n_falling': len(falling),
        'avg_motiv_falling': avg_motiv_falling,
        'avg_motiv_other': avg_motiv_other,
        'motiv_decreased': avg_motiv_falling < avg_motiv_other,
        'result': '✅ ПОДТВЕРЖДЕНО' if avg_motiv_falling < avg_motiv_other else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r5(df_test):
    """R5: Ограничение мотива до 2 для новых запросов"""
    keyword_counts = df_test['keyword'].value_counts()
    new_keywords = keyword_counts[keyword_counts < 10].index.tolist()
    
    if len(new_keywords) < 3:
        return None
    
    df_new = df_test[df_test['keyword'].isin(new_keywords)]
    max_motiv = df_new['motiv'].max()
    
    return {
        'n_new_keywords': len(new_keywords),
        'max_motiv': max_motiv,
        'motiv_limit_ok': max_motiv <= 2,
        'result': '✅ ПОДТВЕРЖДЕНО' if max_motiv <= 2 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r6(df_test):
    """R6: Целевые позиции по частотности"""
    keyword_counts = df_test['keyword'].value_counts()
    high_freq = keyword_counts[keyword_counts > 100].index.tolist()
    low_freq = keyword_counts[keyword_counts <= 100].index.tolist()
    
    results = {}
    
    if high_freq:
        df_high = df_test[df_test['keyword'].isin(high_freq)]
        avg_position_high = df_high[df_high['position'] != -1]['position'].mean()
        results['high_freq_avg_pos'] = avg_position_high
        results['high_freq_target_ok'] = avg_position_high <= 40
    
    if low_freq:
        df_low = df_test[df_test['keyword'].isin(low_freq)]
        avg_position_low = df_low[df_low['position'] != -1]['position'].mean()
        results['low_freq_avg_pos'] = avg_position_low
        results['low_freq_target_ok'] = avg_position_low <= 20
    
    if not results:
        return None
    
    all_ok = all(v for k, v in results.items() if 'target_ok' in k)
    
    return {
        **results,
        'result': '✅ ПОДТВЕРЖДЕНО' if all_ok else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r7(df_test):
    """R7: Эффективность циклов с мотивом 2"""
    df_sorted = df_test.sort_values(['keyword', 'date'])
    results = []
    
    for keyword in df_test['keyword'].unique()[:20]:
        kw_data = df_sorted[df_sorted['keyword'] == keyword]
        if len(kw_data) < 6:
            continue
        
        motiv_pattern = kw_data['motiv'].values[:10]
        for i in range(len(motiv_pattern)-2):
            if motiv_pattern[i] == 2 and motiv_pattern[i+1] == 2 and motiv_pattern[i+2] == 0:
                if i+3 < len(motiv_pattern):
                    delta = kw_data['position'].values[i+3] - kw_data['position'].values[i]
                    results.append(delta)
                break
    
    if not results:
        return None
    
    avg_delta = np.mean(results)
    
    return {
        'n_cycles': len(results),
        'avg_delta': avg_delta,
        'effective': avg_delta < 0,
        'result': '✅ ПОДТВЕРЖДЕНО' if avg_delta < 0 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r8(df_test):
    """R8: Мотив 3 даёт улучшение"""
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    if len(df_analysis) < 20:
        return None
    
    motiv_3 = df_analysis[df_analysis['motiv'] == 3][TARGET_COL]
    motiv_1 = df_analysis[df_analysis['motiv'] == 1][TARGET_COL]
    
    if len(motiv_3) < 3 or len(motiv_1) < 3:
        return None
    
    mean_3 = motiv_3.mean()
    mean_1 = motiv_1.mean()
    
    return {
        'n_motiv_3': len(motiv_3),
        'n_motiv_1': len(motiv_1),
        'mean_3': mean_3,
        'mean_1': mean_1,
        'motiv_3_better': mean_3 < mean_1,
        'result': '✅ ПОДТВЕРЖДЕНО' if mean_3 < mean_1 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r9(df_test):
    """R9: Процентный шаг (мотив растёт)"""
    df_sorted = df_test.sort_values(['keyword', 'date'])
    results = []
    
    for keyword in df_test['keyword'].unique()[:20]:
        kw_data = df_sorted[df_sorted['keyword'] == keyword]
        if len(kw_data) < 5:
            continue
        
        motiv_values = kw_data['motiv'].values[:15]
        for i in range(len(motiv_values)-1):
            if motiv_values[i] > 0 and motiv_values[i+1] > motiv_values[i]:
                increase = motiv_values[i+1] - motiv_values[i]
                results.append({
                    'from': motiv_values[i],
                    'to': motiv_values[i+1],
                    'increase': increase
                })
    
    if not results:
        return None
    
    avg_increase = np.mean([r['increase'] for r in results])
    
    return {
        'n_increases': len(results),
        'avg_increase': avg_increase,
        'pattern_confirmed': avg_increase > 0,
        'result': '✅ ПОДТВЕРЖДЕНО' if avg_increase > 0 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r10(df_test):
    """R10: Откат -30% при отсутствии роста"""
    df_sorted = df_test.sort_values(['keyword', 'date'])
    results = []
    
    for keyword in df_test['keyword'].unique()[:20]:
        kw_data = df_sorted[df_sorted['keyword'] == keyword]
        if len(kw_data) < 5:
            continue
        
        motiv_values = kw_data['motiv'].values[:15]
        position_values = kw_data['position'].values[:15]
        
        for i in range(1, len(motiv_values)):
            if motiv_values[i] < motiv_values[i-1] and motiv_values[i-1] > 0:
                decrease_pct = (motiv_values[i-1] - motiv_values[i]) / motiv_values[i-1]
                if i+1 < len(position_values):
                    delta = position_values[i+1] - position_values[i]
                    results.append({
                        'decrease_pct': decrease_pct,
                        'position_delta': delta,
                        'recovered': delta < 0
                    })
    
    if not results:
        return None
    
    recovered = sum(1 for r in results if r['recovered'])
    
    return {
        'n_rollbacks': len(results),
        'recovered': recovered,
        'pct_recovered': recovered/len(results)*100 if results else 0,
        'result': '✅ ПОДТВЕРЖДЕНО' if recovered/len(results) > 0.3 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r11(df_test):
    """R11: Поддержка снижением на 2-3 единицы"""
    df_sorted = df_test.sort_values(['keyword', 'date'])
    results = []
    
    for keyword in df_test['keyword'].unique()[:20]:
        kw_data = df_sorted[df_sorted['keyword'] == keyword]
        if len(kw_data) < 5:
            continue
        
        last_5 = kw_data.tail(5)
        if len(last_5) >= 3:
            motiv_values = last_5['motiv'].values
            for i in range(len(motiv_values)-1):
                if motiv_values[i+1] < motiv_values[i]:
                    decrease = motiv_values[i] - motiv_values[i+1]
                    if 2 <= decrease <= 3:
                        results.append(decrease)
    
    if not results:
        return None
    
    return {
        'n_support_cases': len(results),
        'avg_decrease': np.mean(results),
        'pattern_confirmed': True,
        'result': '✅ ПОДТВЕРЖДЕНО' if len(results) > 0 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r12(df_test):
    """R12: Мотив не превышает 30% от органического трафика"""
    if 'organic_us' not in df_test.columns:
        return None
    
    df_analysis = df_test.dropna(subset=['organic_us'])
    
    if len(df_analysis) < 10:
        return None
    
    df_analysis = df_analysis.copy()
    df_analysis['motiv_limit'] = df_analysis['organic_us'] * 0.3
    violations = df_analysis[df_analysis['motiv'] > df_analysis['motiv_limit']]
    
    return {
        'n_records': len(df_analysis),
        'n_violations': len(violations),
        'pct_violations': len(violations)/len(df_analysis)*100,
        'result': '✅ ПОДТВЕРЖДЕНО' if len(violations) < len(df_analysis)*0.1 else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r13(df_test):
    """R13: 🆕 Пауза во время фризов"""
    if 'is_freeze' not in df_test.columns:
        return None
    
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    # Мотив во время фризов
    freeze_data = df_analysis[df_analysis['is_freeze'] == 1]
    no_freeze_data = df_analysis[df_analysis['is_freeze'] == 0]
    
    if len(freeze_data) < 10 or len(no_freeze_data) < 10:
        return None
    
    # Средний мотив во время фриза и вне
    avg_motiv_freeze = freeze_data['motiv'].mean()
    avg_motiv_no_freeze = no_freeze_data['motiv'].mean()
    
    # Проверяем: снижается ли мотив во время фризов
    motiv_decreased = avg_motiv_freeze < avg_motiv_no_freeze
    
    # Сравниваем эффективность мотива
    freeze_effect = freeze_data[freeze_data['motiv'] > 0][TARGET_COL].mean() if len(freeze_data[freeze_data['motiv'] > 0]) > 0 else 0
    no_freeze_effect = no_freeze_data[no_freeze_data['motiv'] > 0][TARGET_COL].mean() if len(no_freeze_data[no_freeze_data['motiv'] > 0]) > 0 else 0
    
    # Эффект мотива хуже во время фризов?
    freeze_motiv_less_effective = freeze_effect >= no_freeze_effect
    
    return {
        'n_freeze_records': len(freeze_data),
        'avg_motiv_freeze': avg_motiv_freeze,
        'avg_motiv_no_freeze': avg_motiv_no_freeze,
        'motiv_decreased': motiv_decreased,
        'freeze_effect': freeze_effect,
        'no_freeze_effect': no_freeze_effect,
        'motiv_less_effective_in_freeze': freeze_motiv_less_effective,
        'result': '✅ ПОДТВЕРЖДЕНО' if (motiv_decreased and freeze_motiv_less_effective) else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

def check_rule_r14(df_test):
    """R14: 🆕 Усиление после iOS обновлений"""
    if 'is_week_after_ios_update' not in df_test.columns:
        return None
    
    df_analysis = df_test.dropna(subset=[TARGET_COL])
    
    week_after = df_analysis[df_analysis['is_week_after_ios_update'] == 1]
    normal_days = df_analysis[df_analysis['is_week_after_ios_update'] == 0]
    
    if len(week_after) < 10 or len(normal_days) < 10:
        return None
    
    # Средний мотив
    avg_motiv_update = week_after['motiv'].mean()
    avg_motiv_normal = normal_days['motiv'].mean()
    
    # Среднее изменение позиции
    avg_delta_update = week_after[TARGET_COL].mean()
    avg_delta_normal = normal_days[TARGET_COL].mean()
    
    # Проверяем: мотив растёт в неделю после обновления
    motiv_increased = avg_motiv_update > avg_motiv_normal
    
    # Эффективность продвижения лучше?
    better_effect = avg_delta_update < avg_delta_normal
    
    return {
        'n_week_after': len(week_after),
        'avg_motiv_update': avg_motiv_update,
        'avg_motiv_normal': avg_motiv_normal,
        'avg_delta_update': avg_delta_update,
        'avg_delta_normal': avg_delta_normal,
        'motiv_increased': motiv_increased,
        'better_effect': better_effect,
        'result': '✅ ПОДТВЕРЖДЕНО' if (motiv_increased and better_effect) else '❌ НЕ ПОДТВЕРЖДЕНО'
    }

# =====================================================
# ПРОВЕРКА ВСЕХ ПРАВИЛ
# =====================================================
print("\n" + "="*60)
print("2. ПРОВЕРКА ПРАВИЛ")
print("="*60)

rule_checkers = {
    'R1': check_rule_r1,
    'R2': check_rule_r2,
    'R3': check_rule_r3,
    'R4': check_rule_r4,
    'R5': check_rule_r5,
    'R6': check_rule_r6,
    'R7': check_rule_r7,
    'R8': check_rule_r8,
    'R9': check_rule_r9,
    'R10': check_rule_r10,
    'R11': check_rule_r11,
    'R12': check_rule_r12,
    'R13': check_rule_r13,
    'R14': check_rule_r14
}

rule_results = {}

for rule_id, checker in rule_checkers.items():
    print(f"\n{'='*50}")
    print(f"ПРОВЕРКА: {rule_id} - {rules[rule_id]['name']}")
    print(f"{'='*50}")
    
    result = checker(df_test)
    
    if result:
        rule_results[rule_id] = result
        for key, value in result.items():
            if key != 'result':
                if isinstance(value, float):
                    print(f"  - {key}: {value:.3f}")
                elif isinstance(value, bool):
                    print(f"  - {key}: {'✅ Да' if value else '❌ Нет'}")
                else:
                    print(f"  - {key}: {value}")
        print(f"  - Результат: {result['result']}")
    else:
        print(f"  ⚠️ Недостаточно данных для проверки")

# =====================================================
# СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("3. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*60)

summary_data = []
for rule_id, rule in rules.items():
    if rule_id in rule_results:
        result = rule_results[rule_id]
        summary_data.append({
            'Правило': rule_id,
            'Название': rule['name'][:40],
            'Результат': result.get('result', 'N/A')
        })
    else:
        summary_data.append({
            'Правило': rule_id,
            'Название': rule['name'][:40],
            'Результат': '⚠️ НЕТ ДАННЫХ'
        })

summary_df = pd.DataFrame(summary_data)

def highlight_result(val):
    if '✅' in str(val):
        return 'background-color: #90EE90; color: #006400; font-weight: bold'
    elif '❌' in str(val):
        return 'background-color: #FFB6C1; color: #8B0000; font-weight: bold'
    elif '⚠️' in str(val):
        return 'background-color: #FFD700; color: #8B6914;'
    return ''

if len(summary_df) > 0:
    styled_df = summary_df.style.map(highlight_result, subset=['Результат'])
    styled_df = styled_df.set_caption("📊 Результаты проверки правил (прогноз на 1 день)")
    display(styled_df)

# =====================================================
# ВИЗУАЛИЗАЦИЯ
# =====================================================
print("\n" + "="*60)
print("4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*60)

plots_path = DATA_PATH / "plots"

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Проверка правил из документа (прогноз на 1 день)', fontsize=16, fontweight='bold')

# 1. Подтверждение правил
ax = axes[0, 0]
rule_names = list(rule_results.keys())
accepted = [1 if '✅' in str(r.get('result', '')) else 0 for r in rule_results.values()]
colors = ['green' if a == 1 else 'red' for a in accepted]
bars = ax.barh(rule_names, accepted, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Подтверждение')
ax.set_title('Статус правил')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Не подтверждено', 'Подтверждено'])
ax.grid(True, alpha=0.3, axis='x')

# 2. Процент подтверждения
ax = axes[0, 1]
if accepted:
    total = len(accepted)
    accepted_count = sum(accepted)
    rejected_count = total - accepted_count
    
    sizes = [accepted_count, rejected_count]
    labels = [f'✅ Подтверждено\n({accepted_count})', f'❌ Не подтверждено\n({rejected_count})']
    colors_pie = ['#48bb78', '#fc8181']
    
    ax.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90)
    ax.set_title('Распределение правил')

# 3. R2: Влияние мотива
ax = axes[0, 2]
if 'R2' in rule_results and 'mean_positive' in rule_results['R2']:
    r2 = rule_results['R2']
    categories = ['С мотивом', 'Без мотива']
    values = [r2['mean_positive'], r2['mean_zero']]
    colors = ['green' if v < 0 else 'red' for v in values]
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Среднее изменение позиции')
    ax.set_title(f'R2: Влияние мотива (p={r2["p_value"]:.4f})')
    ax.grid(True, alpha=0.3, axis='y')

# 4. R12: Проверка 30% лимита
ax = axes[1, 0]
if 'R12' in rule_results:
    r12 = rule_results['R12']
    if r12['n_records'] > 0:
        categories = ['В норме', 'Нарушения']
        values = [r12['n_records'] - r12['n_violations'], r12['n_violations']]
        colors = ['green', 'red']
        bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
        ax.set_ylabel('Количество записей')
        ax.set_title(f'R12: Мотив ≤ 30% трафика')
        ax.grid(True, alpha=0.3, axis='y')

# 5. R13: Влияние фризов (НОВОЕ)
ax = axes[1, 1]
if 'R13' in rule_results:
    r13 = rule_results['R13']
    categories = ['Во фризе', 'Вне фриза']
    values = [r13['avg_motiv_freeze'], r13['avg_motiv_no_freeze']]
    colors = ['purple', 'steelblue']
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Средний мотив')
    ax.set_title('R13: Мотив во время фризов')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., val + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# 6. R14: Влияние iOS обновлений (НОВОЕ)
ax = axes[1, 2]
if 'R14' in rule_results:
    r14 = rule_results['R14']
    categories = ['Неделя\nпосле обновления', 'Обычные\nдни']
    values_motiv = [r14['avg_motiv_update'], r14['avg_motiv_normal']]
    values_delta = [r14['avg_delta_update'], r14['avg_delta_normal']]
    
    x = np.arange(len(categories))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, values_motiv, width, label='Средний мотив', 
                   color='orange', alpha=0.7, edgecolor='black')
    bars2 = ax.bar(x + width/2, values_delta, width, label='Δ позиции', 
                   color='steelblue', alpha=0.7, edgecolor='black')
    
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylabel('Значение')
    ax.set_title('R14: Влияние iOS обновлений')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(plots_path / 'rules_1day.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  + График правил сохранён: {plots_path / 'rules_1day.png'}")

# =====================================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# =====================================================
print("\n" + "="*60)
print("5. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

summary_df.to_csv(DATA_PATH / "rules_summary_1day.csv", index=False)
print(f"  + Сводная таблица сохранена: {DATA_PATH / 'rules_summary_1day.csv'}")

# =====================================================
# ИТОГОВЫЕ ВЫВОДЫ
# =====================================================
print("\n" + "="*60)
print("6. ИТОГОВЫЕ ВЫВОДЫ ПО ПРАВИЛАМ")
print("="*60)

if len(summary_df) > 0:
    accepted = summary_df['Результат'].str.contains('✅').sum()
    rejected = summary_df['Результат'].str.contains('❌').sum()
    no_data = summary_df['Результат'].str.contains('⚠️').sum()
    total = len(summary_df)
    
    print(f"""
📊 ОБОБЩЁННЫЕ РЕЗУЛЬТАТЫ ПО ПРАВИЛАМ:

1. СТАТИСТИКА:
   - Всего правил: {total}
   - Подтверждено: {accepted}
   - Не подтверждено: {rejected}
   - Нет данных: {no_data}

2. ПОДТВЕРЖДЁННЫЕ ПРАВИЛА:
""")
    
    for _, row in summary_df.iterrows():
        if '✅' in row['Результат']:
            print(f"   ✅ {row['Правило']}: {row['Название']}")
    
    print(f"\n3. НЕПОДТВЕРЖДЁННЫЕ ПРАВИЛА:")
    for _, row in summary_df.iterrows():
        if '❌' in row['Результат']:
            print(f"   ❌ {row['Правило']}: {row['Название']}")
    
    print(f"\n4. НОВЫЕ ПРАВИЛА (R13, R14):")
    if 'R13' in rule_results:
        print(f"   R13 (Пауза во фризах): {rule_results['R13']['result']}")
    if 'R14' in rule_results:
        print(f"   R14 (Усиление после iOS): {rule_results['R14']['result']}")

print("\n" + "="*80)
print("✅ ШАГ 6 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)


ШАГ 6: ПРОВЕРКА ПРАВИЛ ИЗ ДОКУМЕНТА (ПРОГНОЗ НА 1 ДЕНЬ)

Размер данных: 92,955 записей
Уникальных ключей: 134
Всего признаков: 121

Анализируемый период: 2026-06-20 00:00:00 - 2026-07-15 00:00:00
Записей: 2,978

1. ОПРЕДЕЛЕНИЕ ПРАВИЛ ИЗ ДОКУМЕНТА

Определены правила из документа:
  R1: Базовый цикл продвижения
  R2: Оценка динамики после 2-3 дней
  R3: Увеличение мотива при росте > 10 позиций
  R4: Пропуск при падении
  R5: Ограничение мотива до 2 для новых запросов
  R6: Целевые позиции по частотности
  R7: Максимум 3 цикла с мотивом 2
  R8: Повышение мотива до 3 при росте 50%
  R9: Процентный шаг (15%/10%)
  R10: Откат -30% при отсутствии роста
  R11: Поддержка снижением на 2-3 единицы
  R12: Максимальный мотив 30% от трафика
  R13: 🆕 Пауза во время фризов
  R14: 🆕 Усиление после iOS обновлений

2. ПРОВЕРКА ПРАВИЛ

ПРОВЕРКА: R1 - Базовый цикл продвижения
  ⚠️ Недостаточно данных для проверки

ПРОВЕРКА: R2 - Оценка динамики после 2-3 дней
  - n_positive: 166
  - n_zero: 2812
  - mean

,Правило,Название,Результат
0,R1,Базовый цикл продвижения,⚠️ НЕТ ДАННЫХ
1,R2,Оценка динамики после 2-3 дней,❌ НЕ ПОДТВЕРЖДЕНО
2,R3,Увеличение мотива при росте > 10 позиций,❌ НЕ ПОДТВЕРЖДЕНО
3,R4,Пропуск при падении,✅ ПОДТВЕРЖДЕНО
4,R5,Ограничение мотива до 2 для новых запрос,⚠️ НЕТ ДАННЫХ
5,R6,Целевые позиции по частотности,❌ НЕ ПОДТВЕРЖДЕНО
6,R7,Максимум 3 цикла с мотивом 2,⚠️ НЕТ ДАННЫХ
7,R8,Повышение мотива до 3 при росте 50%,❌ НЕ ПОДТВЕРЖДЕНО
8,R9,Процентный шаг (15%/10%),✅ ПОДТВЕРЖДЕНО
9,R10,Откат -30% при отсутствии роста,❌ НЕ ПОДТВЕРЖДЕНО



4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
  + График правил сохранён: D:\denis\APP\app_new\data\plots\rules_1day.png

5. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
  + Сводная таблица сохранена: D:\denis\APP\app_new\data\rules_summary_1day.csv

6. ИТОГОВЫЕ ВЫВОДЫ ПО ПРАВИЛАМ

📊 ОБОБЩЁННЫЕ РЕЗУЛЬТАТЫ ПО ПРАВИЛАМ:

1. СТАТИСТИКА:
   - Всего правил: 14
   - Подтверждено: 4
   - Не подтверждено: 6
   - Нет данных: 4

2. ПОДТВЕРЖДЁННЫЕ ПРАВИЛА:

   ✅ R4: Пропуск при падении
   ✅ R9: Процентный шаг (15%/10%)
   ✅ R12: Максимальный мотив 30% от трафика
   ✅ R13: 🆕 Пауза во время фризов

3. НЕПОДТВЕРЖДЁННЫЕ ПРАВИЛА:
   ❌ R2: Оценка динамики после 2-3 дней
   ❌ R3: Увеличение мотива при росте > 10 позиций
   ❌ R6: Целевые позиции по частотности
   ❌ R8: Повышение мотива до 3 при росте 50%
   ❌ R10: Откат -30% при отсутствии роста
   ❌ R14: 🆕 Усиление после iOS обновлений

4. НОВЫЕ ПРАВИЛА (R13, R14):
   R13 (Пауза во фризах): ✅ ПОДТВЕРЖДЕНО
   R14 (Усиление после iOS): ❌ НЕ ПОДТВЕРЖДЕНО

✅ ШАГ 6 ЗАВЕРШЕН УСПЕШНО!


# Шаг 7.Вывод


In [8]:
# =====================================================
# ШАГ 7: ОБОБЩЕНИЕ РЕЗУЛЬТАТОВ И РЕКОМЕНДАЦИИ
# =====================================================
print("\n" + "="*80)
print("ШАГ 7: ОБОБЩЕНИЕ РЕЗУЛЬТАТОВ И РЕКОМЕНДАЦИИ")
print("="*80)

# Загрузка данных
df = pd.read_pickle(DATA_PATH / "clean_data.pkl")

print(f"\nРазмер данных: {len(df):,} записей")
print(f"Уникальных ключей: {df['keyword'].nunique()}")
print(f"Всего признаков: {len(df.columns)}")
print(f"Период: {df['date'].min()} - {df['date'].max()}")

from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# =====================================================
# 1. СВОДНАЯ СТАТИСТИКА ПО ДАННЫМ
# =====================================================
print("\n" + "="*60)
print("1. СВОДНАЯ СТАТИСТИКА ПО ДАННЫМ")
print("="*60)

total_records = len(df)
unique_keywords = df['keyword'].nunique()
date_start = df['date'].min()
date_end = df['date'].max()

motiv_mean = df['motiv'].mean()
motiv_median = df['motiv'].median()
motiv_max = df['motiv'].max()
motiv_positive = (df['motiv'] > 0).sum()
motiv_positive_pct = motiv_positive / total_records * 100

position_valid = df[df['position'] != -1]['position']
pos_mean = position_valid.mean()
pos_median = position_valid.median()
pos_min = position_valid.min()
pos_max = position_valid.max()

# Данные по фризам
n_freeze_records = df['is_freeze'].sum() if 'is_freeze' in df.columns else 0
freeze_pct = n_freeze_records / total_records * 100 if total_records > 0 else 0

# Данные по iOS обновлениям
n_ios_updates = df['is_ios_update_day'].sum() if 'is_ios_update_day' in df.columns else 0
n_week_after_ios = df['is_week_after_ios_update'].sum() if 'is_week_after_ios_update' in df.columns else 0

print(f"""
📊 ОБЩАЯ СТАТИСТИКА:
  - Всего записей: {total_records:,}
  - Уникальных ключевых слов: {unique_keywords}
  - Период: {date_start.strftime('%Y-%m-%d')} - {date_end.strftime('%Y-%m-%d')}
  - Дней в периоде: {(date_end - date_start).days}
  - Всего признаков: {len(df.columns)}

📈 МОТИВ:
  - Средний мотив: {motiv_mean:.2f}
  - Медианный мотив: {motiv_median:.2f}
  - Максимальный мотив: {motiv_max:.0f}
  - Записей с мотивом > 0: {motiv_positive:,} ({motiv_positive_pct:.1f}%)

📍 ПОЗИЦИЯ:
  - Средняя позиция: {pos_mean:.2f}
  - Медианная позиция: {pos_median:.2f}
  - Минимальная позиция: {pos_min:.0f}
  - Максимальная позиция: {pos_max:.0f}

❄️ ФРИЗЫ:
  - Записей в периодах фризов: {n_freeze_records:,} ({freeze_pct:.1f}%)

📱 iOS ОБНОВЛЕНИЯ:
  - Дней с обновлениями iOS: {n_ios_updates}
  - Записей в неделю после обновлений: {n_week_after_ios:,}
""")

# =====================================================
# 2. РЕЗУЛЬТАТЫ РАСЧЁТА ЭФФЕКТИВНОСТИ МОТИВА
# =====================================================
print("\n" + "="*60)
print("2. РЕЗУЛЬТАТЫ РАСЧЁТА ЭФФЕКТИВНОСТИ МОТИВА")
print("="*60)

motiv_efficiency_df = None

try:
    motiv_efficiency_df = pd.read_csv(DATA_PATH / "motiv_efficiency.csv")
    
    print(f"\n📊 ЭФФЕКТИВНОСТЬ МОТИВА:")
    print(f"  - Ключей проанализировано: {len(motiv_efficiency_df)}")
    print(f"  - Средняя эффективность: {motiv_efficiency_df['efficiency'].mean():.3f}")
    print(f"  - Медианная эффективность: {motiv_efficiency_df['efficiency'].median():.3f}")
    print(f"  - Максимальная: {motiv_efficiency_df['efficiency'].max():.3f}")
    print(f"  - Минимальная: {motiv_efficiency_df['efficiency'].min():.3f}")
    
    # Ключи с положительной эффективностью
    positive_eff = (motiv_efficiency_df['efficiency'] > 0).sum()
    print(f"  - Ключей с положительной эффективностью: {positive_eff} ({positive_eff/len(motiv_efficiency_df)*100:.1f}%)")
    
    # Топ-10 самых эффективных
    print(f"\n🏆 ТОП-10 САМЫХ ЭФФЕКТИВНЫХ КЛЮЧЕЙ:")
    print(f"  {'Ключ':<35s} {'Эфф.':>8s} {'Ср.мотив':>10s} {'Корр.':>8s}")
    print("  " + "-"*65)
    top_eff = motiv_efficiency_df.sort_values('efficiency', ascending=False).head(10)
    for i, (_, row) in enumerate(top_eff.iterrows(), 1):
        print(f"  {i:2d}. {row['keyword'][:35]:<35s} {row['efficiency']:>8.3f} {row['avg_motiv']:>10.2f} {row['correlation']:>8.3f}")
    
    # Топ-10 самых слабых
    print(f"\n⚠️ ТОП-10 СЛАБЫХ КЛЮЧЕЙ (требуют агрессивного продвижения):")
    print(f"  {'Ключ':<35s} {'Эфф.':>8s} {'Ср.мотив':>10s} {'Корр.':>8s}")
    print("  " + "-"*65)
    worst_eff = motiv_efficiency_df.sort_values('efficiency').head(10)
    for i, (_, row) in enumerate(worst_eff.iterrows(), 1):
        print(f"  {i:2d}. {row['keyword'][:35]:<35s} {row['efficiency']:>8.3f} {row['avg_motiv']:>10.2f} {row['correlation']:>8.3f}")
    
except Exception as e:
    print(f"  ⚠️ Ошибка загрузки эффективности: {e}")

# =====================================================
# 3. РЕЗУЛЬТАТЫ ML МОДЕЛИ
# =====================================================
print("\n" + "="*60)
print("3. РЕЗУЛЬТАТЫ ML МОДЕЛИ (ПРОГНОЗ НА 1 ДЕНЬ)")
print("="*60)

best_model_name = "N/A"
best_mae = 0
best_r2 = 0
best_rmse = 0

try:
    results = pd.read_csv(DATA_PATH / "model_results_1day.csv")
    print("\nСравнение моделей:")
    print(results.to_string(index=False))
    
    best_idx = results['MAE'].idxmin()
    best_model = results.loc[best_idx]
    best_model_name = best_model['Model']
    best_mae = best_model['MAE']
    best_r2 = best_model['R2']
    best_rmse = best_model['RMSE']
    
    print(f"\n🎯 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
    print(f"  - MAE: {best_mae:.4f}")
    print(f"  - RMSE: {best_rmse:.4f}")
    print(f"  - R²: {best_r2:.4f}")
    
    print(f"\n📊 ИНТЕРПРЕТАЦИЯ:")
    print(f"  - MAE = {best_mae:.4f} → модель ошибается в среднем на {best_mae:.2f} позиции")
    print(f"  - R² = {best_r2:.4f} → модель объясняет {best_r2*100:.1f}% вариации")
except Exception as e:
    print(f"  ⚠️ Ошибка загрузки результатов ML: {e}")

# Важность признаков
try:
    imp_df = pd.read_csv(DATA_PATH / "feature_importance_1day.csv")
    print("\n🔍 ТОП-15 ВАЖНЕЙШИХ ПРИЗНАКОВ:")
    print(f"  {'Признак':<35s} {'Важность':>12s} {'Важность %':>12s} {'Группа':>25s}")
    print("  " + "-"*85)
    for i, row in imp_df.head(15).iterrows():
        print(f"  {row['feature'][:35]:<35s} {row['importance']:>12.4f} {row['importance_pct']:>11.2f}% {row['group']:>25s}")
    
    # Группы признаков
    print("\n📊 ВАЖНОСТЬ ПО ГРУППАМ ПРИЗНАКОВ:")
    group_importance = imp_df.groupby('group')['importance'].sum().sort_values(ascending=False)
    total_importance = group_importance.sum()
    for group, imp in group_importance.items():
        pct = (imp / total_importance) * 100
        print(f"  - {group}: {pct:.1f}%")
    
    # Важность признаков событий
    print("\n🎯 ВАЖНОСТЬ ПРИЗНАКОВ СОБЫТИЙ:")
    event_features = ['is_freeze', 'days_to_freeze', 'days_after_freeze',
                      'is_ios_update_day', 'days_after_ios_update', 'is_week_after_ios_update']
    event_imp = imp_df[imp_df['feature'].isin(event_features)]
    if len(event_imp) > 0:
        for _, row in event_imp.iterrows():
            print(f"  - {row['feature']}: {row['importance_pct']:.2f}%")
    else:
        print("  ⚠️ Признаки событий не найдены")
        
except Exception as e:
    print(f"  ⚠️ Ошибка загрузки важности: {e}")

# =====================================================
# 4. РЕЗУЛЬТАТЫ СТАТИСТИЧЕСКИХ ТЕСТОВ
# =====================================================
print("\n" + "="*60)
print("4. РЕЗУЛЬТАТЫ СТАТИСТИЧЕСКИХ ТЕСТОВ")
print("="*60)

hypotheses_summary = None

try:
    hypotheses_summary = pd.read_csv(DATA_PATH / "hypotheses_summary_1day.csv")
    
    print("\nРезультаты проверки гипотез:")
    print("="*80)
    for _, row in hypotheses_summary.iterrows():
        print(f"\n  {row['Гипотеза']}")
        print(f"    Метрика: {row['Метрика']}")
        print(f"    p-value: {row['p-value']}")
        print(f"    Результат: {row['Результат']}")
    
    accepted = (hypotheses_summary['Результат'].str.contains('✅')).sum()
    total = len(hypotheses_summary)
    print(f"\n📊 ИТОГО: {accepted}/{total} гипотез подтверждено ({accepted/total*100:.1f}%)")
    
except Exception as e:
    print(f"  ⚠️ Ошибка загрузки гипотез: {e}")

# =====================================================
# 5. РЕЗУЛЬТАТЫ ПРОВЕРКИ ПРАВИЛ
# =====================================================
print("\n" + "="*60)
print("5. РЕЗУЛЬТАТЫ ПРОВЕРКИ ПРАВИЛ")
print("="*60)

rules_summary = None

try:
    rules_summary = pd.read_csv(DATA_PATH / "rules_summary_1day.csv")
    
    print("\nРезультаты проверки правил:")
    print(rules_summary.to_string(index=False))
    
    accepted_rules = (rules_summary['Результат'].str.contains('✅')).sum()
    total_rules = len(rules_summary)
    print(f"\n📊 ИТОГО: {accepted_rules}/{total_rules} правил подтверждено")
    
except Exception as e:
    print(f"  ⚠️ Ошибка загрузки правил: {e}")

# =====================================================
# 6. АНАЛИЗ ФРИЗОВ И ОБНОВЛЕНИЙ iOS
# =====================================================
print("\n" + "="*60)
print("6. АНАЛИЗ ФРИЗОВ И ОБНОВЛЕНИЙ iOS")
print("="*60)

# Загружаем справочники
try:
    freeze_df = pd.read_csv(DATA_PATH / "freeze_periods.csv")
    ios_df = pd.read_csv(DATA_PATH / "ios_updates.csv")
    
    print(f"\n❄️ ПЕРИОДЫ ФРИЗОВ ({len(freeze_df)} периодов):")
    print(f"  {'Начало':<12s} {'Конец':<12s} {'Название':<30s} {'Записей':>10s} {'Δ позиции':>12s}")
    print("  " + "-"*80)
    for _, row in freeze_df.iterrows():
        df_period = df[(df['date'] >= pd.to_datetime(row['start'])) & 
                       (df['date'] <= pd.to_datetime(row['end']))]
        if len(df_period) > 0:
            avg_delta = df_period['position_delta_1d'].mean() if 'position_delta_1d' in df_period.columns else 0
            print(f"  {row['start']:<12s} {row['end']:<12s} {row['name'][:30]:<30s} {len(df_period):>10,} {avg_delta:>+12.3f}")
    
    print(f"\n📱 iOS ОБНОВЛЕНИЯ ({len(ios_df)} обновлений):")
    print(f"  {'Дата':<12s} {'Версия':<15s} {'Записей':>10s} {'Δ неделя':>12s}")
    print("  " + "-"*55)
    for _, row in ios_df.iterrows():
        df_period = df[(df['date'] >= pd.to_datetime(row['date'])) & 
                       (df['date'] <= pd.to_datetime(row['date']) + pd.Timedelta(days=7))]
        if len(df_period) > 0:
            avg_delta = df_period['position_delta_1d'].mean() if 'position_delta_1d' in df_period.columns else 0
            print(f"  {row['date']:<12s} {row['version']:<15s} {len(df_period):>10,} {avg_delta:>+12.3f}")
    
    # Анализ влияния фризов
    print(f"\n📊 СРАВНЕНИЕ: ФРИЗЫ vs ОБЫЧНЫЕ ДНИ")
    df_freeze = df[df['is_freeze'] == 1]
    df_no_freeze = df[df['is_freeze'] == 0]
    
    print(f"  Во фризах:")
    print(f"    - Записей: {len(df_freeze):,}")
    print(f"    - Средний мотив: {df_freeze['motiv'].mean():.2f}")
    print(f"    - Δ позиции (1д): {df_freeze['position_delta_1d'].mean():+.3f}")
    
    print(f"  Вне фризов:")
    print(f"    - Записей: {len(df_no_freeze):,}")
    print(f"    - Средний мотив: {df_no_freeze['motiv'].mean():.2f}")
    print(f"    - Δ позиции (1д): {df_no_freeze['position_delta_1d'].mean():+.3f}")
    
    # Анализ iOS обновлений
    print(f"\n📊 СРАВНЕНИЕ: НЕДЕЛЯ ПОСЛЕ iOS vs ОБЫЧНЫЕ ДНИ")
    df_after_ios = df[df['is_week_after_ios_update'] == 1]
    df_normal = df[df['is_week_after_ios_update'] == 0]
    
    print(f"  Неделя после обновления:")
    print(f"    - Записей: {len(df_after_ios):,}")
    print(f"    - Средний мотив: {df_after_ios['motiv'].mean():.2f}")
    print(f"    - Δ позиции (1д): {df_after_ios['position_delta_1d'].mean():+.3f}")
    
    print(f"  Обычные дни:")
    print(f"    - Записей: {len(df_normal):,}")
    print(f"    - Средний мотив: {df_normal['motiv'].mean():.2f}")
    print(f"    - Δ позиции (1д): {df_normal['position_delta_1d'].mean():+.3f}")
    
except Exception as e:
    print(f"  ⚠️ Ошибка анализа событий: {e}")

# =====================================================
# 7. ИТОГОВЫЙ ДАШБОРД
# =====================================================
print("\n" + "="*60)
print("7. ИТОГОВЫЙ ДАШБОРД")
print("="*60)

# Собираем все метрики
dashboard_data = {
    'records': total_records,
    'keywords': unique_keywords,
    'period': f"{date_start.strftime('%Y-%m-%d')} - {date_end.strftime('%Y-%m-%d')}",
    'avg_position': pos_mean,
    'avg_motiv': motiv_mean,
    'best_model': best_model_name,
    'best_mae': best_mae,
    'best_r2': best_r2,
    'hypotheses_confirmed': 0,
    'hypotheses_total': 0,
    'rules_confirmed': 0,
    'rules_total': 0,
    'avg_efficiency': 0,
    'freeze_pct': freeze_pct,
    'ios_updates_count': 0
}

if hypotheses_summary is not None:
    dashboard_data['hypotheses_confirmed'] = (hypotheses_summary['Результат'].str.contains('✅')).sum()
    dashboard_data['hypotheses_total'] = len(hypotheses_summary)

if rules_summary is not None:
    dashboard_data['rules_confirmed'] = (rules_summary['Результат'].str.contains('✅')).sum()
    dashboard_data['rules_total'] = len(rules_summary)

if motiv_efficiency_df is not None:
    dashboard_data['avg_efficiency'] = motiv_efficiency_df['efficiency'].mean()

try:
    dashboard_data['ios_updates_count'] = len(ios_df)
except:
    pass

# Отображаем дашборд
display(HTML(f"""
<div style="padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; color: white; margin: 20px 0;">
    <h2 style="margin-top: 0; color: white; text-align: center;">📊 ИТОГОВЫЙ ДАШБОРД АНАЛИЗА</h2>
    <p style="text-align: center; opacity: 0.9; margin-bottom: 20px;">Прогноз на 1 день | Учёт фризов и iOS обновлений</p>
    
    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap: 15px; margin-top: 20px;">
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 28px; font-weight: bold;">{dashboard_data['records']:,}</div>
            <div style="font-size: 12px; opacity: 0.9;">Всего записей</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 28px; font-weight: bold;">{dashboard_data['keywords']}</div>
            <div style="font-size: 12px; opacity: 0.9;">Ключевых слов</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 28px; font-weight: bold;">{dashboard_data['avg_position']:.1f}</div>
            <div style="font-size: 12px; opacity: 0.9;">Средняя позиция</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 28px; font-weight: bold;">{dashboard_data['avg_motiv']:.2f}</div>
            <div style="font-size: 12px; opacity: 0.9;">Средний мотив</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 28px; font-weight: bold;">{dashboard_data['avg_efficiency']:.2f}</div>
            <div style="font-size: 12px; opacity: 0.9;">Ср. эффективность мотива</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['best_mae']:.4f}</div>
            <div style="font-size: 12px; opacity: 0.9;">MAE модели</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['best_r2']:.4f}</div>
            <div style="font-size: 12px; opacity: 0.9;">R² модели</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['hypotheses_confirmed']}/{dashboard_data['hypotheses_total']}</div>
            <div style="font-size: 12px; opacity: 0.9;">Гипотез подтверждено</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['rules_confirmed']}/{dashboard_data['rules_total']}</div>
            <div style="font-size: 12px; opacity: 0.9;">Правил подтверждено</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['freeze_pct']:.1f}%</div>
            <div style="font-size: 12px; opacity: 0.9;">Записей во фризах</div>
        </div>
        <div style="background: rgba(255,255,255,0.15); padding: 15px; border-radius: 10px; text-align: center;">
            <div style="font-size: 22px; font-weight: bold;">{dashboard_data['ios_updates_count']}</div>
            <div style="font-size: 12px; opacity: 0.9;">iOS обновлений</div>
        </div>
    </div>
</div>
"""))

# =====================================================
# 8. РЕКОМЕНДАЦИИ ДЛЯ ASO
# =====================================================
print("\n" + "="*60)
print("8. РЕКОМЕНДАЦИИ ДЛЯ ASO")
print("="*60)

display(HTML("""
<div style="padding: 20px; background: #f8f9fa; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin-top: 0;">🎯 ПРАКТИЧЕСКИЕ РЕКОМЕНДАЦИИ</h3>
    
    <div style="margin: 15px 0; padding: 15px; background: #fff3cd; border-left: 4px solid #ffc107; border-radius: 5px;">
        <h4 style="margin: 0; color: #856404;">🔴 КРИТИЧЕСКИЕ РЕКОМЕНДАЦИИ</h4>
        <ul style="margin: 10px 0 0 0;">
            <li><b>Используйте расчёт требуемого мотива</b> для планирования продвижения</li>
            <li><b>Прогноз на 1 день</b> даёт наиболее точные краткосрочные оценки</li>
            <li><b>Ставьте паузу во время фризов</b> — продвижение не работает</li>
            <li><b>Усиливайте активность после iOS обновлений</b> — пик активности пользователей</li>
        </ul>
    </div>
    
    <div style="margin: 15px 0; padding: 15px; background: #d1ecf1; border-left: 4px solid #17a2b8; border-radius: 5px;">
        <h4 style="margin: 0; color: #0c5460;">🟡 ВАЖНЫЕ РЕКОМЕНДАЦИИ</h4>
        <ul style="margin: 10px 0 0 0;">
            <li><b>Эффективность мотива</b> разная по ключам — учитывайте при планировании</li>
            <li><b>Для ключей с низкой эффективностью</b> используйте агрессивный мотив</li>
            <li><b>Для ключей с высокой эффективностью</b> достаточно умеренного мотива</li>
            <li><b>Учитывайте правило R12</b> (мотив ≤ 30% от трафика)</li>
        </ul>
    </div>
    
    <div style="margin: 15px 0; padding: 15px; background: #d4edda; border-left: 4px solid #28a745; border-radius: 5px;">
        <h4 style="margin: 0; color: #155724;">🟢 РЕКОМЕНДУЕМЫЕ ДЕЙСТВИЯ</h4>
        <ul style="margin: 10px 0 0 0;">
            <li><b>Проверяйте гипотезы</b> на свежих данных еженедельно</li>
            <li><b>Переобучайте модель</b> каждую неделю</li>
            <li><b>Используйте LLM-ассистента</b> для генерации новых гипотез</li>
            <li><b>Планируйте продвижение</b> с учётом календаря фризов</li>
            <li><b>Готовьте контент заранее</b> для усиления после фризов</li>
        </ul>
    </div>
</div>
"""))

# =====================================================
# 9. РЕКОМЕНДАЦИИ ПО ЦЕЛЕВЫМ ПОЗИЦИЯМ
# =====================================================
print("\n" + "="*60)
print("9. РЕКОМЕНДАЦИИ ПО ЦЕЛЕВЫМ ПОЗИЦИЯМ")
print("="*60)

if motiv_efficiency_df is not None:
    # Функция расчёта требуемого мотива
    def calc_required_motiv(keyword, target_pos):
        kw_data = df[df['keyword'] == keyword]
        if len(kw_data) == 0:
            return None
        
        valid_pos = kw_data[kw_data['position'] != -1]
        current_pos = valid_pos['position'].iloc[-1] if len(valid_pos) > 0 else -1
        
        eff_row = motiv_efficiency_df[motiv_efficiency_df['keyword'] == keyword]
        if len(eff_row) == 0:
            return None
        
        efficiency = eff_row.iloc[0]['efficiency']
        position_diff = current_pos - target_pos
        
        if position_diff <= 0:
            return 0
        
        if efficiency > 0.1:
            required = position_diff / efficiency
        else:
            required = position_diff * 0.5
        
        return min(max(required, 1), 30)
    
    # Топ-10 ключей и их целевые позиции
    top10 = df['keyword'].value_counts().head(10).index.tolist()
    
    print("\n🎯 РЕКОМЕНДАЦИИ ПО ЦЕЛЕВЫМ ПОЗИЦИЯМ (топ-10 ключей):")
    print(f"\n  {'Ключ':<30s} {'Текущая':>10s} {'Топ-10':>10s} {'Топ-20':>10s} {'Топ-50':>10s}")
    print("  " + "-"*75)
    
    for keyword in top10:
        kw_data = df[df['keyword'] == keyword]
        valid_pos = kw_data[kw_data['position'] != -1]
        current = valid_pos['position'].iloc[-1] if len(valid_pos) > 0 else -1
        
        m_top10 = calc_required_motiv(keyword, 10)
        m_top20 = calc_required_motiv(keyword, 20)
        m_top50 = calc_required_motiv(keyword, 50)
        
        m_top10_str = f"{m_top10:.0f}" if m_top10 is not None else "-"
        m_top20_str = f"{m_top20:.0f}" if m_top20 is not None else "-"
        m_top50_str = f"{m_top50:.0f}" if m_top50 is not None else "-"
        
        print(f"  {keyword[:30]:<30s} {current:>10.1f} {m_top10_str:>10s} {m_top20_str:>10s} {m_top50_str:>10s}")
    
    # Сохраняем рекомендации
    recommendations_data = []
    for keyword in df['keyword'].unique():
        kw_data = df[df['keyword'] == keyword]
        valid_pos = kw_data[kw_data['position'] != -1]
        current = valid_pos['position'].iloc[-1] if len(valid_pos) > 0 else -1
        
        recommendations_data.append({
            'keyword': keyword,
            'current_position': current,
            'required_motiv_top10': calc_required_motiv(keyword, 10),
            'required_motiv_top20': calc_required_motiv(keyword, 20),
            'required_motiv_top50': calc_required_motiv(keyword, 50)
        })
    
    recommendations_df = pd.DataFrame(recommendations_data)
    recommendations_df.to_csv(DATA_PATH / "motiv_recommendations.csv", index=False)
    print(f"\n  + Рекомендации сохранены: {DATA_PATH / 'motiv_recommendations.csv'}")

# =====================================================
# 10. КАЛЕНДАРЬ СОБЫТИЙ ДЛЯ ПЛАНИРОВАНИЯ
# =====================================================
print("\n" + "="*60)
print("10. КАЛЕНДАРЬ СОБЫТИЙ ДЛЯ ПЛАНИРОВАНИЯ")
print("="*60)

try:
    # Следующий фриз
    from datetime import datetime
    today = datetime.now()
    
    future_freezes = []
    for _, row in freeze_df.iterrows():
        start = pd.to_datetime(row['start'])
        if start > today:
            future_freezes.append({
                'start': row['start'],
                'end': row['end'],
                'name': row['name'],
                'days_until': (start - today).days
            })
    
    if future_freezes:
        next_freeze = min(future_freezes, key=lambda x: x['days_until'])
        print(f"\n❄️ СЛЕДУЮЩИЙ ФРИЗ:")
        print(f"  - Название: {next_freeze['name']}")
        print(f"  - Начало: {next_freeze['start']}")
        print(f"  - Конец: {next_freeze['end']}")
        print(f"  - Дней до начала: {next_freeze['days_until']}")
        
        if next_freeze['days_until'] <= 7:
            print(f"\n  ⚠️ ВНИМАНИЕ: Фриз начнётся менее чем через 7 дней!")
            print(f"  📌 Рекомендация: приостановить увеличение мотива")
    else:
        print(f"\n❄️ Все запланированные фризы прошли")
    
    # Последнее iOS обновление
    past_updates = []
    for _, row in ios_df.iterrows():
        date = pd.to_datetime(row['date'])
        if date <= today:
            past_updates.append({
                'date': row['date'],
                'version': row['version'],
                'days_ago': (today - date).days
            })
    
    if past_updates:
        last_update = max(past_updates, key=lambda x: x['date'])
        print(f"\n📱 ПОСЛЕДНЕЕ iOS ОБНОВЛЕНИЕ:")
        print(f"  - Версия: iOS {last_update['version']}")
        print(f"  - Дата: {last_update['date']}")
        print(f"  - Дней назад: {last_update['days_ago']}")
        
        if last_update['days_ago'] <= 7:
            print(f"\n  ✅ Неделя после iOS обновления — пик активности!")
            print(f"  📌 Рекомендация: усилить продвижение")
    
    # Следующее iOS обновление (прогноз по паттерну)
    print(f"\n📅 ПАТТЕРН iOS ОБНОВЛЕНИЙ:")
    print(f"  - Средний интервал между обновлениями: ~30-45 дней")
    print(f"  - Обычно крупные обновления: март, июнь, сентябрь")
    
except Exception as e:
    print(f"  ⚠️ Ошибка: {e}")

# =====================================================
# 11. ИТОГОВЫЙ ВЕРДИКТ
# =====================================================
print("\n" + "="*60)
print("11. ИТОГОВЫЙ ВЕРДИКТ")
print("="*60)

# Расчёт итоговых метрик
if hypotheses_summary is not None:
    h_confirmed = (hypotheses_summary['Результат'].str.contains('✅')).sum()
    h_total = len(hypotheses_summary)
    h_pct = h_confirmed/h_total*100 if h_total > 0 else 0
else:
    h_confirmed, h_total, h_pct = 0, 0, 0

if rules_summary is not None:
    r_confirmed = (rules_summary['Результат'].str.contains('✅')).sum()
    r_total = len(rules_summary)
    r_pct = r_confirmed/r_total*100 if r_total > 0 else 0
else:
    r_confirmed, r_total, r_pct = 0, 0, 0

avg_eff = motiv_efficiency_df['efficiency'].mean() if motiv_efficiency_df is not None else 0

print(f"""
🎯 ОСНОВНЫЕ РЕЗУЛЬТАТЫ АНАЛИЗА:

1. ДАННЫЕ:
   ✅ Обработано {total_records:,} записей
   ✅ {unique_keywords} уникальных ключевых слов
   ✅ Период: {date_start.strftime('%Y-%m-%d')} - {date_end.strftime('%Y-%m-%d')}
   ✅ Всего признаков: {len(df.columns)}

2. ЭФФЕКТИВНОСТЬ МОТИВА:
   ✅ Рассчитана для {len(motiv_efficiency_df) if motiv_efficiency_df is not None else 'N/A'} ключей
   ✅ Средняя эффективность: {avg_eff:.3f}
   ✅ Возможность расчёта требуемого мотива под целевую позицию

3. ML МОДЕЛИ:
   ✅ Лучшая модель: {best_model_name}
   ✅ MAE: {best_mae:.4f} (прогноз на 1 день)
   ✅ R²: {best_r2:.4f}
   ✅ Использовано расширенных признаков (включая события)

4. ГИПОТЕЗЫ:
   ✅ Подтверждено: {h_confirmed}/{h_total} ({h_pct:.1f}%)
   ✅ Включены новые гипотезы H7 (фризы) и H8 (iOS обновления)

5. ПРАВИЛА:
   📋 Проверено правил: {r_total}
   📊 Подтверждено: {r_confirmed}/{r_total} ({r_pct:.1f}%)
   ✅ Включены новые правила R13 (пауза во фризах) и R14 (усиление после iOS)

6. СОБЫТИЯ:
   ❄️ Периодов фризов: {len(freeze_df) if 'freeze_df' in locals() else 0}
   📱 iOS обновлений: {len(ios_df) if 'ios_df' in locals() else 0}
   ✅ Учтены в признаках ML модели

7. КЛЮЧЕВЫЕ ВЫВОДЫ:
   🔴 Следите за позицией — это главный фактор
   🟡 Мотив важен, но эффективность разная по ключам
   🟢 Используйте расчёт требуемого мотива для планирования
   📅 Прогноз на 1 день даёт точные краткосрочные оценки
   ❄️ Ставьте паузу во время фризов
   📱 Усиливайтесь после iOS обновлений

8. НОВЫЕ ВОЗМОЖНОСТИ:
   ✨ Расчёт требуемого мотива для целевой позиции
   ✨ Эффективность мотива по каждому ключу
   ✨ Расширенные статистические признаки
   ✨ Учёт фризов и iOS обновлений
   ✨ LLM-проверка гипотез по каждому ключу
   ✨ Календарь событий для планирования
""")

print("\n" + "="*80)
print("🎉 АНАЛИЗ ЗАВЕРШЕН! ВСЕ ШАГИ ВЫПОЛНЕНЫ!")
print("="*80)

# =====================================================
# 12. СОХРАНЕНИЕ ИТОГОВОГО ОТЧЕТА
# =====================================================
print("\n" + "="*60)
print("12. СОХРАНЕНИЕ ИТОГОВОГО ОТЧЕТА")
print("="*60)

import json
from datetime import datetime

# Собираем информацию о событиях для отчёта
freeze_info = []
if 'freeze_df' in locals():
    for _, row in freeze_df.iterrows():
        freeze_info.append({
            'start': row['start'],
            'end': row['end'],
            'name': row['name']
        })

ios_info = []
if 'ios_df' in locals():
    for _, row in ios_df.iterrows():
        ios_info.append({
            'date': row['date'],
            'version': row['version']
        })

final_report = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'target_metric': 'position_delta_1d',
    'data_summary': {
        'total_records': int(total_records),
        'unique_keywords': int(unique_keywords),
        'total_features': int(len(df.columns)),
        'period_start': date_start.strftime('%Y-%m-%d'),
        'period_end': date_end.strftime('%Y-%m-%d'),
        'motiv_mean': float(motiv_mean),
        'position_mean': float(pos_mean)
    },
    'motiv_efficiency': {
        'n_keywords': int(len(motiv_efficiency_df)) if motiv_efficiency_df is not None else 0,
        'mean_efficiency': float(avg_eff),
        'median_efficiency': float(motiv_efficiency_df['efficiency'].median()) if motiv_efficiency_df is not None else 0,
        'max_efficiency': float(motiv_efficiency_df['efficiency'].max()) if motiv_efficiency_df is not None else 0,
        'min_efficiency': float(motiv_efficiency_df['efficiency'].min()) if motiv_efficiency_df is not None else 0
    },
    'model_results': {
        'best_model': best_model_name,
        'mae': float(best_mae),
        'rmse': float(best_rmse),
        'r2': float(best_r2),
        'target_metric': 'position_delta_1d'
    },
    'hypotheses': {
        'confirmed': int(h_confirmed),
        'total': int(h_total),
        'confirmation_rate': float(h_pct),
        'new_hypotheses': [
            'H7: Влияние фризов на продвижение',
            'H8: Влияние iOS обновлений'
        ]
    },
    'rules': {
        'confirmed': int(r_confirmed),
        'total': int(r_total),
        'confirmation_rate': float(r_pct),
        'new_rules': [
            'R13: Пауза во время фризов',
            'R14: Усиление после iOS обновлений'
        ]
    },
    'events': {
        'freeze_periods': freeze_info,
        'ios_updates': ios_info,
        'n_freeze_records': int(n_freeze_records),
        'freeze_pct': float(freeze_pct),
        'n_ios_updates': int(n_ios_updates),
        'n_week_after_ios': int(n_week_after_ios)
    },
    'recommendations': {
        'critical': [
            'Используйте расчёт требуемого мотива для планирования',
            'Прогноз на 1 день даёт наиболее точные оценки',
            'Ставьте паузу во время фризов',
            'Усиливайте активность после iOS обновлений'
        ],
        'important': [
            'Эффективность мотива разная по ключам',
            'Для низкой эффективности — агрессивный мотив',
            'Для высокой эффективности — умеренный мотив',
            'Учитывайте правило R12 (мотив ≤ 30% трафика)'
        ],
        'recommended': [
            'Проверяйте гипотезы на свежих данных еженедельно',
            'Переобучайте модель каждую неделю',
            'Используйте LLM для генерации гипотез',
            'Планируйте продвижение с учётом календаря событий'
        ]
    }
}

# Сохраняем
with open(DATA_PATH / 'final_report_1day.json', 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)

print(f"  + Итоговый отчёт сохранён: {DATA_PATH / 'final_report_1day.json'}")

# Сохраняем также краткий отчёт
brief_summary = {
    'analysis_date': final_report['analysis_date'],
    'total_records': total_records,
    'unique_keywords': unique_keywords,
    'avg_position': round(pos_mean, 2),
    'avg_motiv': round(motiv_mean, 2),
    'best_model': best_model_name,
    'best_mae': round(best_mae, 4),
    'best_r2': round(best_r2, 4),
    'avg_efficiency': round(avg_eff, 3),
    'hypotheses_confirmed': f"{h_confirmed}/{h_total}",
    'rules_confirmed': f"{r_confirmed}/{r_total}",
    'freeze_periods': len(freeze_info),
    'ios_updates': len(ios_info)
}

with open(DATA_PATH / 'brief_summary_1day.json', 'w', encoding='utf-8') as f:
    json.dump(brief_summary, f, indent=2, ensure_ascii=False)

print(f"  + Краткий отчёт сохранён: {DATA_PATH / 'brief_summary_1day.json'}")

# =====================================================
# 13. ФИНАЛЬНАЯ СВОДКА ФАЙЛОВ
# =====================================================
print("\n" + "="*60)
print("13. СОЗДАННЫЕ ФАЙЛЫ")
print("="*60)

files_created = [
    ('clean_data.pkl', 'Очищенные данные с признаками'),
    ('clean_data.csv', 'Очищенные данные (CSV)'),
    ('motiv_efficiency.csv', 'Эффективность мотива по ключам'),
    ('keyword_stats.csv', 'Статистика по ключам'),
    ('model_results_1day.csv', 'Результаты ML моделей'),
    ('best_model_xgboost_1day.pkl', 'Лучшая ML модель'),
    ('scaler_1day.pkl', 'Scaler для модели'),
    ('feature_importance_1day.csv', 'Важность признаков'),
    ('hypotheses_summary_1day.csv', 'Результаты гипотез'),
    ('rules_summary_1day.csv', 'Результаты правил'),
    ('freeze_periods.csv', 'Справочник фризов'),
    ('ios_updates.csv', 'Справочник iOS обновлений'),
    ('motiv_recommendations.csv', 'Рекомендации по мотиву'),
    ('final_report_1day.json', 'Итоговый отчёт'),
    ('brief_summary_1day.json', 'Краткий отчёт'),
]

print(f"\n  {'Файл':<40s} {'Описание':<40s}")
print("  " + "-"*80)
for filename, description in files_created:
    file_path = DATA_PATH / filename
    exists = "✅" if file_path.exists() else "❌"
    print(f"  {exists} {filename:<38s} {description:<40s}")

# Графики
print(f"\n  📊 ГРАФИКИ:")
plots_path = DATA_PATH / "plots"
for plot_file in sorted(plots_path.glob("*.png")):
    print(f"    ✅ {plot_file.name}")

print("\n" + "="*80)
print("✅ ШАГ 7 ЗАВЕРШЕН УСПЕШНО!")
print("="*80)

print(f"""
🎉 АНАЛИЗ ПОЛНОСТЬЮ ЗАВЕРШЕН!

Все результаты сохранены в папку: {DATA_PATH}

Для просмотра результатов:
1. Откройте файл final_report_1day.json — полный отчёт
2. Откройте brief_summary_1day.json — краткая сводка
3. Графики в папке plots/ — все визуализации
4. Запустите сервис (python run.py) для интерактивного анализа

Следующие шаги:
- Используйте сервис для расчёта требуемого мотива
- Проверяйте гипотезы через LLM-ассистента
- Планируйте продвижение с учётом календаря событий
""")


ШАГ 7: ОБОБЩЕНИЕ РЕЗУЛЬТАТОВ И РЕКОМЕНДАЦИИ

Размер данных: 92,955 записей
Уникальных ключей: 134
Всего признаков: 121
Период: 2023-12-28 00:00:00 - 2026-07-15 00:00:00

1. СВОДНАЯ СТАТИСТИКА ПО ДАННЫМ

📊 ОБЩАЯ СТАТИСТИКА:
  - Всего записей: 92,955
  - Уникальных ключевых слов: 134
  - Период: 2023-12-28 - 2026-07-15
  - Дней в периоде: 930
  - Всего признаков: 121

📈 МОТИВ:
  - Средний мотив: 0.77
  - Медианный мотив: 0.00
  - Максимальный мотив: 200
  - Записей с мотивом > 0: 14,279 (15.4%)

📍 ПОЗИЦИЯ:
  - Средняя позиция: 46.93
  - Медианная позиция: 30.00
  - Минимальная позиция: 0
  - Максимальная позиция: 249

❄️ ФРИЗЫ:
  - Записей в периодах фризов: 14,604 (15.7%)

📱 iOS ОБНОВЛЕНИЯ:
  - Дней с обновлениями iOS: 1296
  - Записей в неделю после обновлений: 9,059


2. РЕЗУЛЬТАТЫ РАСЧЁТА ЭФФЕКТИВНОСТИ МОТИВА

📊 ЭФФЕКТИВНОСТЬ МОТИВА:
  - Ключей проанализировано: 103
  - Средняя эффективность: 9.338
  - Медианная эффективность: 4.402
  - Максимальная: 60.380
  - Минимальная: -20.102



8. РЕКОМЕНДАЦИИ ДЛЯ ASO



9. РЕКОМЕНДАЦИИ ПО ЦЕЛЕВЫМ ПОЗИЦИЯМ

🎯 РЕКОМЕНДАЦИИ ПО ЦЕЛЕВЫМ ПОЗИЦИЯМ (топ-10 ключей):

  Ключ                              Текущая     Топ-10     Топ-20     Топ-50
  ---------------------------------------------------------------------------
  sound amplifier                       2.0          0          0          0
  ear spy                               8.0          0          0          0
  listening device                     16.0          2          0          0
  louder volume                        28.0          3          2          0
  speaker booster                     106.0         15         13          9
  hearing amplifier                     1.0          0          0          0
  amplificador de volumen              40.0         16         11          0
  sound booster                        34.0          7          4          0
  hearing amplifier free                9.0          0          0          0
  amp me                                8.0          0       